# Windbreaker — Phase 1, 2 & 3: Data, XGBoost & Alerting System

**Project:** Windbreaker Recommendation Engine (XGBoost)

**Purpose:** Consolidate historical Meta advertising campaign data across
multiple clients (Client A, Client B, Client C, Client D and Client E) to support the
development of a machine learning recommendation system for campaign
optimisation.

**the agency partner x Bond University Capstone | DTSC71-304**

This notebook covers **Phase 1** (data), **Phase 2** (modelling) and
**Phase 3** (alerting & recommendations) of the Windbreaker pipeline:

| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Data Loading & KPI Guide Integration (daily + monthly) |
| 3 | Data Cleaning & Standardisation |
| 4 | Exploratory Data Analysis (EDA) - all 5 clients |
| 5 | Cross-Client Feature Alignment |
| 6 | Goal-Specific KPI Analysis (CPL & CPT focus) |
| 7 | Recommendation Threshold Framework |
| 8 | Feature Engineering (for XGBoost) |
| 9 | Data Export for Modelling |
| 10 | Data Quality Summary & Cross-Client EDA |
| 11 | **Phase 2 — XGBoost: Binary model (PRIMARY) + 4-class (comparison)** |
| 12 | **Feature Importance & Model Evaluation** |
| 13 | **Phase 3 — Binary Sigmoid-to-4-Class Mapping (the advisor's framework)** |
| 14 | **Alerting System Recommendations (documentation for the agency partner)** |
| 15 | **Baseline Metrics & Recommendations Report** |
| 16 | **Project Constraints & Documentation** |

> **Clients:** Client A (Client A) · Client C (Client C) · Client B (Client B) · Client E · Client D
>
> **Key goals:** Leads (Client A, Client D) · Purchases (Client C, Client B) · Reach (Client E)

> **Model approach (agreed March 13 session with the advisors):**
> XGBoost rather than a neural network, due to small dataset size (~500-1000
> rows). This also enables feature importance extraction, a key deliverable
> for the agency partner.
>
> **PRIMARY approach (advisors, March 13):**
> Train a BINARY classifier (Scale / Do Not Scale) and use the sigmoid
> probability output to derive 4 recommendation tiers via thresholds.
> Rationale: a small dataset makes training a 4-category model directly
> problematic. A binary model is more robust, and its output probabilities
> can be used to define custom classification tiers.
>
> **SECONDARY approach (comparison only):**
> Direct 4-class XGBoost with softprob. Retained for comparison but NOT
> the primary deliverable per mentor guidance.

> **Data update (March 17):**
> The client analyst provided daily-level CSV exports for Client A, Client C, and Client B (split by
> year/period). Client D and Client E daily xlsx files were already available
> from the previous batch.
>
> **Data strategy per client (based on daily assessment):**
> - Client C: Daily CSVs as primary (purchase data now available)
> - Client B: Daily CSVs as primary (transforms from EDA-only to trainable)
> - Client D: Daily xlsx as primary
> - Client E: Daily xlsx as primary
> - Client A: Daily CSVs as primary for CPL model (986 rows with leads). Monthly xlsx retained as comparison only.

> **Targets (agreed March 13 internal session):**
>
> CPL (Cost Per Lead) for lead-generation clients (Client A, Client D)
>
> CPT (Cost Per Purchase) for e-commerce clients (Client C, Client B)
>
> These serve both as classification targets and as threshold controls.

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install/Import Required Packages

In [ ]:
# Install/Import Required Packages

# Install any missing packages
# !pip install openpyxl xgboost scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# ── Phase 2 imports ───────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)
from sklearn.calibration import CalibratedClassifierCV
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost version:", xgb.__version__)
except ImportError:
    XGBOOST_AVAILABLE = False
    print("WARNING: XGBoost not installed. Run: !pip install xgboost")

# ── Windbreaker Colour Palette ─────────────────────────────────────────────

WB_PALETTE = {
    # Per-client colours (blue-green gradient, dark → light)
    'CLIENT_A':      '#1A5276',   # deep navy blue
    'CLIENT_C':     '#1F8B8B',   # teal
    'CLIENT_B':       '#239B89',   # muted teal-green
    'CLIENT_E': '#52BE9E',   # medium seafoam
    'CLIENT_D':    '#7DCEA0',   # soft sage green
    # Recommendation colours (semantic)
    'Scale Up':  '#28B463',  # green  — strong efficiency
    'Maintain':  '#2E86C1',  # blue   — on-target
    'Optimise':  '#E6A817',  # muted gold
    'Pause':     '#C0392B',  # muted red
    # Alert severity
    'CRITICAL':  '#C0392B',  # red
    'WARNING':   '#E6A817',  # amber
    'INFO':      '#2E86C1',  # blue
    # General-purpose
    'primary':   '#1A5276',
    'secondary': '#1F8B8B',
    'accent':    '#239B89',
    'light':     '#52BE9E',
    'grid':      '#D5E8D4',
    'ref_line':  '#5D6D7E',
}
# List form for seaborn / multi-client charts
WB_CLIENT_COLORS = [WB_PALETTE['CLIENT_A'], WB_PALETTE['CLIENT_C'], WB_PALETTE['CLIENT_B'],
                    WB_PALETTE['CLIENT_E'], WB_PALETTE['CLIENT_D']]

# Apply global plotting style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.prop_cycle': plt.cycler(color=WB_CLIENT_COLORS),
})
sns.set_palette(WB_CLIENT_COLORS)

# Custom blue-green colourmap for heatmaps
WB_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'windbreaker', ['#A3E4D7', '#1F8B8B', '#1A5276']
)
# Diverging version (for correlation matrices)
WB_CMAP_DIV = mcolors.LinearSegmentedColormap.from_list(
    'windbreaker_div', ['#CB4335', '#F7F7F7', '#1A5276']
)

print("Imports and Windbreaker palette loaded")

## 2. Data Loading & KPI Guide Integration

### Data Overview

the client analyst provided daily-level CSV exports on March 17 for Client A, Client C, and Client B
(split by year/period). Client D and Client E daily xlsx files were already
available from the previous batch.

**Data strategy per client:**
- Client C and Client B: Daily CSVs concatenated as primary training data
- Client D and Client E: Daily xlsx as primary
- Client A: Client A: Daily CSVs as primary for CPL model (986 rows with leads). Monthly xlsx retained as comparison only.

| Client | Primary Source | Format | Target Metric | Training |
|--------|---------------|--------|---------------|----------|
|Client A | Daily CSVs (8 files) | CSV | CPL | Yes (daily, 986 rows with leads) |
| Client C | Daily CSVs (4 files) | CSV | CPT | Yes (daily) |
| Client B | Daily CSVs (4 files) | CSV | CPT | Yes (daily) |
| Client D | Daily xlsx | Excel | CPL | Evaluate volume |
| Client E | Daily xlsx | Excel | CPM | EDA + validation |


In [ ]:
# ── File paths ─────────────────────────────────────────────────────────────
# All files sit flat in the Windbreaker folder (no subfolders).

DATA_DIR = Path('/content/drive/MyDrive/<project-folder>

# CLIENT_A monthly (COMPARISON ONLY — daily is now primary for CPL model)
CLIENT_A_MONTHLY = DATA_DIR / 'CLIENT_A-050223-050326.xlsx'

# Daily CSV files (the client analyst's March 17 exports — concatenated per client)
DAILY_CSV_FILES = {
    'CLIENT_A_daily': [   # PRIMARY for CPL model (986 rows with leads)
        DATA_DIR / 'CLIENT_A-2023-Untitled-report-Feb-18-2023-to-Dec-31-2023.csv',
        DATA_DIR / 'CLIENT_A-2024-1-Untitled-report-Jan-1-2024-to-Mar-31-2024.csv',
        DATA_DIR / 'CLIENT_A-2024-2-Untitled-report-Apr-1-2024-to-Jun-30-2024.csv',
        DATA_DIR / 'CLIENT_A-2024-3-Untitled-report-Jul-1-2024-to-Sep-30-2024.csv',
        DATA_DIR / 'CLIENT_A-2024-4-Untitled-report-Oct-1-2024-to-Dec-31-2024.csv',
        DATA_DIR / 'CLIENT_A-2025-1-Untitled-report-Jan-1-2025-to-Jul-31-2025.csv',
        DATA_DIR / 'CLIENT_A-2025-2-Untitled-report-Aug-1-2025-to-Dec-31-2025.csv',
        DATA_DIR / 'CLIENT_A-2026-Untitled-report-Jan-1-2026-to-Mar-17-2026.csv',
    ],
    'CLIENT_C': [
        DATA_DIR / 'CLIENT_C-2023-Untitled-report-Feb-18-2023-to-Dec-31-2023.csv',
        DATA_DIR / 'CLIENT_C-2024-Untitled-report-Jan-1-2024-to-Dec-31-2024.csv',
        DATA_DIR / 'CLIENT_C-2025-Untitled-report-Jan-1-2025-to-Dec-31-2025.csv',
        DATA_DIR / 'CLIENT_C-2026-Untitled-report-Jan-1-2026-to-Mar-17-2026.csv',
    ],
    'CLIENT_B': [
        DATA_DIR / 'CLIENT_B-2023-Untitled-report-Feb-18-2023-to-Dec-31-2023.csv',
        DATA_DIR / 'CLIENT_B-2024-Untitled-report-Jan-1-2024-to-Dec-31-2024.csv',
        DATA_DIR / 'CLIENT_B-2025-Untitled-report-Jan-1-2025-to-Dec-31-2025.csv',
        DATA_DIR / 'CLIENT_B-2026-Untitled-report-Jan-1-2026-to-Mar-17-2026.csv',
    ],
}

# Daily xlsx files (previous batch — single files per client)
DAILY_XLSX_FILES = {
    'CLIENT_D':   DATA_DIR / 'CLIENT_D-Untitled-report-Feb-13-2023-to-Mar-13-2026.xlsx',
    'CLIENT_E': DATA_DIR / 'client_e-Untitled-report-Feb-13-2023-to-Mar-13-2026.xlsx',
}

KPI_FILE = DATA_DIR / 'Meta_Ads_KPI_Guide.xlsx'

# ── Loader: monthly Excel ─────────────────────────────────────────────────
def load_monthly_excel(name, path, header_row=0, sheet='Raw Data Report'):
    """Load a client monthly Excel file with auto-header detection."""
    df = pd.read_excel(path, sheet_name=sheet, header=header_row)
    df = df.dropna(how='all')

    expected_cols = {'Amount spent (AUD)', 'Month', 'Impressions', 'Reach'}
    if len(set(df.columns) & expected_cols) == 0:
        for try_header in [0, 1, 2, 3]:
            if try_header == header_row:
                continue
            df_retry = pd.read_excel(path, sheet_name=sheet, header=try_header)
            df_retry = df_retry.dropna(how='all')
            if len(set(df_retry.columns) & expected_cols) > 0:
                df = df_retry
                print(f"  [{name}] Auto-detected header at row {try_header}")
                break

    df['_client'] = name
    df['_granularity'] = 'monthly'
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols  [MONTHLY xlsx]")
    return df

# ── Loader: daily CSVs (concatenate multiple files) ────────────────────────
def load_daily_csvs(name, file_list):
    """Load and concatenate multiple daily CSV files for a client."""
    frames = []
    for csv_path in file_list:
        try:
            df = pd.read_csv(csv_path)
            df = df.dropna(how='all')
            fname = Path(csv_path).name
            print(f"    {fname}: {len(df)} rows")
            frames.append(df)
        except FileNotFoundError:
            print(f"    NOT FOUND: {Path(csv_path).name}")
        except Exception as e:
            print(f"    ERROR: {Path(csv_path).name} — {e}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        combined['_client'] = name
        combined['_granularity'] = 'daily'
        print(f"  {name}: {len(combined)} rows total  [DAILY CSV, {len(frames)} files]")
        return combined
    else:
        print(f"  {name}: No daily CSV data loaded")
        return pd.DataFrame()

# ── Loader: daily Excel ───────────────────────────────────────────────────
def load_daily_xlsx(name, path, sheet='Raw Data Report'):
    """Load a daily Excel file with auto-header detection."""
    try:
        df = pd.read_excel(path, sheet_name=sheet, header=0)
        df = df.dropna(how='all')

        expected_cols = {'Amount spent (AUD)', 'Day', 'Impressions', 'Reach'}
        if len(set(df.columns) & expected_cols) == 0:
            for try_header in [1, 2, 3]:
                df_retry = pd.read_excel(path, sheet_name=sheet, header=try_header)
                df_retry = df_retry.dropna(how='all')
                if len(set(df_retry.columns) & expected_cols) > 0:
                    df = df_retry
                    print(f"  [{name}] Auto-detected header at row {try_header}")
                    break

        df['_client'] = name
        df['_granularity'] = 'daily'
        print(f"  {name}: {len(df)} rows x {len(df.columns)} cols  [DAILY xlsx]")
        return df
    except FileNotFoundError:
        print(f"  {name}: FILE NOT FOUND — {path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"  {name}: ERROR — {e}")
        return pd.DataFrame()

# ── Load all datasets ──────────────────────────────────────────────────────
print("=" * 70)
print("LOADING DATASETS (v7 — daily primary where available)")
print("=" * 70)

raw = {}

# CLIENT_A: monthly as primary (leads are sparse at daily level)
print("\n--- CLIENT_A (monthly — comparison only) ---")
raw['CLIENT_A'] = load_monthly_excel('CLIENT_A', CLIENT_A_MONTHLY)

# CLIENT_C: daily CSVs as primary (purchase data now available)
print("\n--- CLIENT_C (daily CSVs — primary) ---")
raw['CLIENT_C'] = load_daily_csvs('CLIENT_C', DAILY_CSV_FILES['CLIENT_C'])

# CLIENT_B: daily CSVs as primary (transforms from EDA-only to trainable)
print("\n--- CLIENT_B (daily CSVs — primary) ---")
raw['CLIENT_B'] = load_daily_csvs('CLIENT_B', DAILY_CSV_FILES['CLIENT_B'])

# CLIENT_D: daily xlsx as primary
print("\n--- CLIENT_D (daily xlsx) ---")
raw['CLIENT_D'] = load_daily_xlsx('CLIENT_D', DAILY_XLSX_FILES['CLIENT_D'])

# CLIENT_E: daily xlsx as primary
print("\n--- CLIENT_E (daily xlsx) ---")
raw['CLIENT_E'] = load_daily_xlsx('CLIENT_E', DAILY_XLSX_FILES['CLIENT_E'])

# CLIENT_A daily: loaded separately for EDA/spend analysis only
print("\n--- CLIENT_A daily (PRIMARY for CPL model) ---")
client_a_daily_eda = load_daily_csvs('CLIENT_A', DAILY_CSV_FILES['CLIENT_A_daily'])

# ── KPI reference guide ────────────────────────────────────────────────────
kpi_guide = pd.read_excel(KPI_FILE, sheet_name='Meta Ads KPIs')
print(f"\n  KPI Guide: {len(kpi_guide)} KPIs loaded")

# ── Loading summary ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DATA LOADING SUMMARY")
print("=" * 70)
for name, df in raw.items():
    gran = df['_granularity'].iloc[0] if len(df) > 0 else 'empty'
    print(f"  {name:<10} {gran:<10} {len(df):>6} rows  [PRIMARY]")
if len(client_a_daily_eda) > 0:
    print(f"  {'CLIENT_A_daily':<10} {'daily':<10} {len(client_a_daily_eda):>6} rows  [EDA/SPEND ONLY]")
print("\nAll files loaded")

### 2.1  KPI Guide — Identify Critical Metrics

the client analyst's KPI guide classifies 34 Meta Ads metrics across 7 categories
(Delivery, Traffic, Engagement, Conversions, Ecommerce Funnel, Lead
Generation, Diagnostics).  Each KPI is flagged as **Critical (Y/N)**.

We parse this here to build a list of critical KPIs that drives feature
selection for the XGBoost model, ensuring the model prioritises the
metrics that the agency partner considers most important.

In [ ]:
# ── Parse KPI guide ─────────────────────────────────────────────────────────
print("=" * 60)
print("KPI GUIDE — OVERVIEW")
print("=" * 60)
print(f"  Total KPIs:  {len(kpi_guide)}")
print(f"  Categories:  {kpi_guide['Category'].nunique()}  ({', '.join(kpi_guide['Category'].unique())})")
print(f"  Critical:    {(kpi_guide['Critical'] == 'Y').sum()} Y  /  {(kpi_guide['Critical'] == 'N').sum()} N")

# ── Split into critical vs non-critical ─────────────────────────────────────
critical_kpis = kpi_guide[kpi_guide['Critical'] == 'Y']['KPI'].tolist()
non_critical_kpis = kpi_guide[kpi_guide['Critical'] == 'N']['KPI'].tolist()

print(f"\nCritical KPIs ({len(critical_kpis)}):")
for _, row in kpi_guide[kpi_guide['Critical'] == 'Y'].iterrows():
    print(f"  Y  [{row['Category']:<20}]  {row['KPI']}")

print(f"\nNon-critical KPIs ({len(non_critical_kpis)}):")
for _, row in kpi_guide[kpi_guide['Critical'] == 'N'].iterrows():
    print(f"  N  [{row['Category']:<20}]  {row['KPI']}")

# ── Map the client analyst's KPI names → our cleaned column names ───────────────────────
KPI_NAME_TO_COL = {
    'Impressions':                  'impressions',
    'Reach':                        'reach',
    'Frequency':                    'frequency',
    'CPM':                          'cpm',
    'Link Clicks':                  'link_clicks',
    'CTR (Click-Through Rate)':     'ctr_link',
    'Unique CTR':                   'ctr_all',
    'CPC (Cost Per Click)':         'cpc_link',
    'Landing Page Views':           None,
    'Cost per Landing Page View':   None,
    'Post Engagement':              None,
    'Engagement Rate':              None,
    'Video Views':                  None,
    'ThruPlays':                    None,
    'Cost per ThruPlay':            None,
    'Average Watch Time':           None,
    'Conversions':                  'results',
    'Cost per Result (CPA)':        'cost_per_result',
    'Conversion Rate':              None,
    'Conversion Value':             'purchase_value',
    'ROAS (Return on Ad Spend)':    'roas',
    'Add to Cart':                  None,
    'Cost per Add to Cart':         None,
    'Initiate Checkout':            None,
    'Cost per Initiate Checkout':   None,
    'Purchases':                    'purchases',
    'Cost per Purchase':            'cost_per_purchase',
    'Leads':                        'all_leads',
    'Cost per Lead (CPL)':          'cost_per_lead',
    'Lead Form Completion Rate':    None,
    'Quality Ranking':              'quality_ranking',
    'Engagement Rate Ranking':      'engagement_rate_ranking',
    'Conversion Rate Ranking':      'conversion_rate_ranking',
    'Outbound Clicks':              'outbound_ctr',
}

# Build the critical column list for feature engineering
CRITICAL_FEATURE_COLS = []
unmapped = []

for kpi_name in critical_kpis:
    col = KPI_NAME_TO_COL.get(kpi_name)
    if col is not None:
        CRITICAL_FEATURE_COLS.append(col)
    else:
        unmapped.append(kpi_name)

CRITICAL_FEATURE_COLS = list(dict.fromkeys(CRITICAL_FEATURE_COLS))  # deduplicate

print(f"\n{'=' * 60}")
print(f"CRITICAL KPIs MAPPED TO DATASET COLUMNS ({len(CRITICAL_FEATURE_COLS)})")
print(f"{'=' * 60}")
for col in CRITICAL_FEATURE_COLS:
    orig = [k for k, v in KPI_NAME_TO_COL.items() if v == col]
    print(f"  {col:<25} <- {orig[0] if orig else '?'}")

if unmapped:
    print(f"\nCritical KPIs not in current data ({len(unmapped)}):")
    for k in unmapped:
        print(f"  [not available]  {k}")
    print("  These are valid KPIs but not present in the Meta exports we received.")

KPI_GUIDE_AVAILABLE = True

## 3. Data Cleaning & Standardisation

### Cleaning steps applied:
1. Rename columns to snake_case for consistency
2. **Auto-detect granularity:** parse `Day` column (daily) or `Month` column
   (monthly) into unified `date_start` / `date_end` fields
3. Strip whitespace from string columns
4. Replace '-' placeholder strings with NaN
5. Cast numeric columns to float
6. Flag and handle nulls
7. Remove rows with zero spend (no campaign activity)
8. Derive `cost_per_purchase` for Client C if not in export (spend / purchases)

In [ ]:
# ── Column name mapping ─────────────────────────────────────────────────────
COL_MAP = {
    'Campaign name':                    'campaign_name',
    'Ad set name':                      'adset_name',
    'Ad name':                          'ad_name',
    'Day':                              'day',             # daily files
    'Month':                            'month',           # monthly files
    'Delivery status':                  'delivery_status',
    'Delivery level':                   'delivery_level',
    'Result Type':                      'result_type',
    'Results':                          'results',
    'Cost per result':                  'cost_per_result',
    'Amount spent (AUD)':               'spend',
    'Impressions':                      'impressions',
    'Reach':                            'reach',
    'Frequency':                        'frequency',
    'CPM (cost per 1,000 impressions)': 'cpm',
    'Link clicks':                      'link_clicks',
    'Attribution setting':              'attribution_setting',
    'Quality ranking':                  'quality_ranking',
    'Engagement rate ranking':          'engagement_rate_ranking',
    'Conversion rate ranking':          'conversion_rate_ranking',
    'Leads':                            'leads',
    'Cost per All Leads (AUD)':         'cost_per_lead',
    'All Leads':                        'all_leads',
    'Clicks (all)':                     'clicks_all',
    'CTR (all)':                        'ctr_all',
    'CTR (link click-through rate)':    'ctr_link',
    'CPC (cost per link click)':        'cpc_link',
    'CPC (all)':                        'cpc_all',
    'Reporting starts':                 'date_start',
    'Reporting ends':                   'date_end',
    # CLIENT_C / CLIENT_B specific
    'Purchases':                        'purchases',
    'Cost per purchase':                'cost_per_purchase',
    'Purchase ROAS (return on ad spend)':'roas',
    'Purchases conversion value':       'purchase_value',
    'Outbound CTR (click-through rate)':'outbound_ctr',
    'Results ROAS':                     'results_roas',
}

NUMERIC_COLS = [
    'spend', 'impressions', 'reach', 'frequency', 'cpm', 'link_clicks',
    'results', 'cost_per_result', 'leads', 'cost_per_lead', 'all_leads',
    'clicks_all', 'ctr_all', 'ctr_link', 'cpc_link', 'cpc_all',
    'purchases', 'cost_per_purchase', 'roas', 'purchase_value', 'outbound_ctr',
]

def clean_client(df, name):
    """
    Clean a single client DataFrame. Handles both daily and monthly files.
    """
    granularity = df['_granularity'].iloc[0] if '_granularity' in df.columns else 'unknown'

    # Drop fully empty columns
    df = df.dropna(axis=1, how='all').copy()

    # Rename columns using the mapping
    df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

    # Drop unnamed index columns (CLIENT_B artefact)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

    # ── Fallback column detection ──────────────────────────────────────────
    if 'spend' not in df.columns:
        spend_candidates = [c for c in df.columns if 'spent' in str(c).lower() or 'spend' in str(c).lower()]
        if spend_candidates:
            df = df.rename(columns={spend_candidates[0]: 'spend'})
            print(f"  [{name}] Fallback: mapped '{spend_candidates[0]}' -> 'spend'")

    if 'impressions' not in df.columns:
        imp_candidates = [c for c in df.columns if 'impression' in str(c).lower()]
        if imp_candidates:
            df = df.rename(columns={imp_candidates[0]: 'impressions'})

    if 'cpm' not in df.columns:
        cpm_candidates = [c for c in df.columns if 'cpm' in str(c).lower()]
        if cpm_candidates:
            df = df.rename(columns={cpm_candidates[0]: 'cpm'})

    # Replace '-' with NaN in ranking-style columns
    for col in ['quality_ranking', 'engagement_rate_ranking', 'conversion_rate_ranking']:
        if col in df.columns:
            df[col] = df[col].replace('-', np.nan)

    # ── Date parsing (granularity-aware) ──────────────────────────────────
    # Priority 1: 'day' column (daily files)
    if 'day' in df.columns:
        df['date_start'] = pd.to_datetime(df['day'], errors='coerce')
        df['date_end']   = df['date_start']  # same day
        print(f"  [{name}] Parsed daily dates from 'Day' column")

    # Priority 2: 'date_start'/'date_end' already exist (Reporting starts/ends)
    elif 'date_start' in df.columns:
        df['date_start'] = pd.to_datetime(df['date_start'], errors='coerce')
        if 'date_end' in df.columns:
            df['date_end'] = pd.to_datetime(df['date_end'], errors='coerce')

    # Priority 3: 'month' column (monthly format: "2024-01-01 - 2024-01-31")
    if 'month' in df.columns:
        if 'date_start' not in df.columns or df['date_start'].isna().all():
            df['date_start'] = pd.to_datetime(
                df['month'].str.split(' - ').str[0], errors='coerce'
            )
            df['date_end'] = pd.to_datetime(
                df['month'].str.split(' - ').str[1], errors='coerce'
            )
            print(f"  [{name}] Parsed monthly dates from 'Month' column")

    # Cast numerics
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Derive year_month for grouping (works for both daily and monthly)
    if 'date_start' in df.columns:
        df['year_month'] = df['date_start'].dt.to_period('M')

    # Derive day_of_week for daily data
    if granularity == 'daily' and 'date_start' in df.columns:
        df['day_of_week'] = df['date_start'].dt.day_name()
        df['day_of_week_num'] = df['date_start'].dt.dayofweek  # 0=Mon, 6=Sun

    # Derive cost_per_purchase if not in export (CLIENT_C daily CSVs)
    if 'purchases' in df.columns and 'cost_per_purchase' not in df.columns:
        purchases = pd.to_numeric(df.get('purchases', 0), errors='coerce')
        if (purchases > 0).any():
            df['cost_per_purchase'] = df['spend'] / purchases.replace(0, np.nan)
            derived_count = df['cost_per_purchase'].notna().sum()
            print(f"  [{name}] Derived cost_per_purchase: {derived_count} rows")

    # Drop zero-spend rows (no campaign activity)
    if 'spend' in df.columns:
        before = len(df)
        df = df[df['spend'].notna() & (df['spend'] > 0)]
        dropped = before - len(df)
        if dropped > 0:
            print(f"  [{name}] Dropped {dropped} zero/null spend rows")
    else:
        print(f"  [{name}] WARNING: 'spend' column not found")
        print(f"           Available columns: {list(df.columns[:15])}")

    # Tag client and granularity
    df['_client'] = name
    df['_granularity'] = granularity
    return df


print("Cleaning datasets")
cleaned = {}
for name, df in raw.items():
    cleaned[name] = clean_client(df, name)
    gran = cleaned[name]['_granularity'].iloc[0] if len(cleaned[name]) > 0 else 'empty'
    print(f"  {name}: {len(cleaned[name])} rows after cleaning  [{gran}]")

# Also clean CLIENT_A daily for EDA use
if len(client_a_daily_eda) > 0:
    client_a_daily_cleaned = clean_client(client_a_daily_eda, 'CLIENT_A')
    print(f"  CLIENT_A_daily (EDA): {len(client_a_daily_cleaned)} rows after cleaning  [daily]")
else:
    client_a_daily_cleaned = pd.DataFrame()

print("\nCleaning complete")


In [ ]:
# ── Data summary ────────────────────────────────────────────────────────────
print("\n=== DATA SUMMARY (v7) ===")
for name, df in cleaned.items():
    gran = df['_granularity'].iloc[0] if len(df) > 0 else 'N/A'
    date_min = df['date_start'].min() if 'date_start' in df.columns else 'N/A'
    date_max = df['date_start'].max() if 'date_start' in df.columns else 'N/A'
    print(f"  {name:<10} {gran:<10} {len(df):>6} rows   {date_min} -> {date_max}")
if len(client_a_daily_cleaned) > 0:
    d_min = client_a_daily_cleaned['date_start'].min() if 'date_start' in client_a_daily_cleaned.columns else 'N/A'
    d_max = client_a_daily_cleaned['date_start'].max() if 'date_start' in client_a_daily_cleaned.columns else 'N/A'
    print(f"  {'CLIENT_A_daily':<10} {'daily':<10} {len(client_a_daily_cleaned):>6} rows   {d_min} -> {d_max}  [EDA only]")

In [ ]:
# ── Null summary ────────────────────────────────────────────────────────────
print("\n=== NULL SUMMARY AFTER CLEANING ===\n")
for name, df in cleaned.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"{'~'*40}")
    print(f"  {name}  ({len(df)} rows)")
    if len(nulls) == 0:
        print("  No nulls")
    else:
        for col, count in nulls.items():
            pct = count / len(df) * 100
            print(f"  {col:<35} {count:>4} nulls ({pct:.1f}%)")


## 4. Exploratory Data Analysis (EDA)

We analyse **each client independently**, focusing on their **primary KPI**
and the key drivers of campaign cost-efficiency.

Every client receives a dedicated analysis section, regardless of data volume.
For small datasets (Client B, Client E, Client D), we note the limitations but still
present whatever patterns exist.

In [ ]:
# ── 4.1  Spend over time - all clients ─────────────────────────────────────
fig, axes = plt.subplots(1, len(cleaned), figsize=(18, 4), sharey=False)

for ax, (name, df), color in zip(axes, cleaned.items(), WB_CLIENT_COLORS):
    if 'date_start' not in df.columns or df['date_start'].isna().all():
        ax.set_title(f'{name}\n(no date data)')
        continue
    monthly = df.groupby(df['date_start'].dt.to_period('M'))['spend'].sum()
    monthly.index = monthly.index.to_timestamp()
    ax.bar(monthly.index, monthly.values, color=color, alpha=0.65, width=20)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Month')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Monthly Ad Spend by Client', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.2  Client A — Lead Generation Deep Dive

- **Goal:** Leads / CPL  
- **Industry:** Dental healthcare  
- **Data volume:** Primary training candidate

In [ ]:
# ── 4.2  CLIENT_A — Lead generation deep dive ────────────────────────────────────
client_a = cleaned['CLIENT_A'].copy()
C_CLIENT_A = WB_PALETTE['CLIENT_A']

# Compute CPL if not already clean
client_a['cpl_calc'] = client_a['spend'] / client_a['all_leads'].replace(0, np.nan)
client_a = client_a[client_a['all_leads'] > 0]  # Only rows where leads were recorded

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# CPL distribution
axes[0].hist(client_a['cpl_calc'].dropna(), bins=20, color=C_CLIENT_A, edgecolor='white')
axes[0].axvline(client_a['cpl_calc'].median(), color=WB_PALETTE['ref_line'], linestyle='--',
                label=f"Median: ${client_a['cpl_calc'].median():.0f}")
axes[0].set_title('CLIENT_A — CPL Distribution')
axes[0].set_xlabel('Cost Per Lead ($)')
axes[0].legend()

# CPL over time
monthly_cpl = client_a.groupby(client_a['date_start'].dt.to_period('M'))['cpl_calc'].mean()
monthly_cpl.index = monthly_cpl.index.to_timestamp()
axes[1].plot(monthly_cpl.index, monthly_cpl.values, marker='o', color=C_CLIENT_A, linewidth=2)
axes[1].set_title('CLIENT_A — Avg CPL Over Time')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('CPL ($)')
axes[1].tick_params(axis='x', rotation=45)

# Spend vs Leads scatter
axes[2].scatter(client_a['spend'], client_a['all_leads'], alpha=0.6, color=C_CLIENT_A, edgecolor='white')
axes[2].set_title('CLIENT_A — Spend vs Leads')
axes[2].set_xlabel('Spend ($)')
axes[2].set_ylabel('Leads')

plt.tight_layout()
plt.show()

print("CLIENT_A CPL stats:")
print(client_a['cpl_calc'].describe().round(2).to_string())

### 4.3  Client C - Purchase / ROAS Deep Dive

- **Goal:** Purchases / ROAS  
- **Industry:** Art supplies ecommerce  
- **Data volume:** Primary training candidate

In [ ]:
# ── 4.3  CLIENT_C — Purchase / ROAS deep dive ───────────────────────────────────
client_c = cleaned['CLIENT_C'].copy()
C_CLIENT_C = WB_PALETTE['CLIENT_C']

client_c_has_purchases = 'purchases' in client_c.columns and client_c['purchases'].notna().sum() > 0

if client_c_has_purchases:
    client_c_purch = client_c[client_c['purchases'].notna() & (client_c['purchases'] > 0)]
    print(f"CLIENT_C: {len(client_c_purch)} rows with purchase data")
else:
    client_c_purch = client_c
    print("CLIENT_C: No purchase columns in this export. Showing general metrics.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Cost per purchase OR CPM distribution
if 'cost_per_purchase' in client_c_purch.columns and client_c_purch['cost_per_purchase'].notna().sum() > 0:
    axes[0].hist(client_c_purch['cost_per_purchase'].dropna(), bins=25, color=C_CLIENT_C, edgecolor='white')
    axes[0].axvline(client_c_purch['cost_per_purchase'].median(), color=WB_PALETTE['ref_line'],
                    linestyle='--', linewidth=1.2, alpha=0.65,
                    label=f"Median: ${client_c_purch['cost_per_purchase'].median():.0f}")
    axes[0].set_title('CLIENT_C — Cost per Purchase (CPT) Distribution')
    axes[0].set_xlabel('Cost per Purchase ($)')
    axes[0].legend()
elif 'cpm' in client_c.columns and client_c['cpm'].notna().sum() > 0:
    axes[0].hist(client_c['cpm'].dropna(), bins=25, color=C_CLIENT_C, edgecolor='white')
    axes[0].axvline(client_c['cpm'].median(), color=WB_PALETTE['ref_line'],
                    linestyle='--', linewidth=1.2, alpha=0.65,
                    label=f"Median CPM: ${client_c['cpm'].median():.2f}")
    axes[0].set_title('CLIENT_C — CPM Distribution (no purchase data)')
    axes[0].set_xlabel('CPM ($)')
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, 'No Cost per Purchase or CPM data', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=11, color='grey')
    axes[0].set_title('CLIENT_C — Cost Metrics')

# Panel 2: ROAS distribution OR spend distribution
if 'roas' in client_c_purch.columns and client_c_purch['roas'].notna().sum() > 0:
    axes[1].hist(client_c_purch['roas'].dropna(), bins=25, color=C_CLIENT_C, edgecolor='white')
    axes[1].axvline(client_c_purch['roas'].median(), color=WB_PALETTE['ref_line'],
                    linestyle='--', linewidth=1.2, alpha=0.65,
                    label=f"Median ROAS: {client_c_purch['roas'].median():.2f}x")
    axes[1].set_title('CLIENT_C — ROAS Distribution')
    axes[1].set_xlabel('ROAS')
    axes[1].legend()
else:
    axes[1].hist(client_c['spend'].dropna(), bins=25, color=C_CLIENT_C, edgecolor='white')
    axes[1].set_title('CLIENT_C — Spend Distribution')
    axes[1].set_xlabel('Spend ($)')

# Panel 3: Spend vs Purchases OR Impressions vs CTR
if client_c_has_purchases:
    axes[2].scatter(client_c_purch['spend'], client_c_purch['purchases'], alpha=0.6,
                    color=C_CLIENT_C, edgecolor='white')
    axes[2].set_title('CLIENT_C — Spend vs Purchases')
    axes[2].set_xlabel('Spend ($)')
    axes[2].set_ylabel('Purchases')
else:
    if 'ctr_link' in client_c.columns and client_c['ctr_link'].notna().sum() > 0:
        axes[2].scatter(client_c['impressions'], client_c['ctr_link'], alpha=0.6,
                        color=C_CLIENT_C, edgecolor='white')
        axes[2].set_title('CLIENT_C — Impressions vs CTR')
        axes[2].set_xlabel('Impressions')
        axes[2].set_ylabel('CTR (Link)')
    else:
        axes[2].text(0.5, 0.5, 'Limited metrics available', ha='center', va='center',
                     transform=axes[2].transAxes, fontsize=11, color='grey')

plt.tight_layout()
plt.show()

print("CLIENT_C key metrics:")
for col in ['spend', 'purchases', 'cost_per_purchase', 'roas', 'cpm']:
    if col in client_c.columns:
        print(f"  {col}: median={client_c[col].median():.2f}, mean={client_c[col].mean():.2f}")


### 4.4  Client B - Purchase / ROAS Deep Dive


- **Goal:** Purchases / ROAS  
- **Industry:** Lighting ecommerce

In [ ]:
# ── 4.4  CLIENT_B — Client B deep dive ─────────────────────────────────
client_b = cleaned['CLIENT_B'].copy()
C_CLIENT_B = WB_PALETTE['CLIENT_B']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(client_b['spend'].dropna(), bins=10, color=C_CLIENT_B, edgecolor='white')
axes[0].axvline(client_b['spend'].median(), color=WB_PALETTE['ref_line'], linestyle='--',
                label=f"Median: ${client_b['spend'].median():.0f}")
axes[0].set_title('CLIENT_B — Spend Distribution')
axes[0].set_xlabel('Spend ($)')
axes[0].legend()

if 'roas' in client_b.columns and client_b['roas'].notna().sum() > 0:
    client_b_roas = client_b[client_b['roas'].notna() & (client_b['roas'] > 0)]
    axes[1].hist(client_b_roas['roas'].dropna(), bins=10, color=C_CLIENT_B, edgecolor='white')
    axes[1].axvline(client_b_roas['roas'].median(), color=WB_PALETTE['ref_line'], linestyle='--',
                    label=f"Median: {client_b_roas['roas'].median():.2f}x")
    axes[1].set_title('CLIENT_B — ROAS Distribution')
    axes[1].set_xlabel('ROAS')
    axes[1].legend()
elif 'cost_per_purchase' in client_b.columns and client_b['cost_per_purchase'].notna().sum() > 0:
    client_b_cpp = client_b[client_b['cost_per_purchase'].notna()]
    axes[1].hist(client_b_cpp['cost_per_purchase'].dropna(), bins=10, color=C_CLIENT_B, edgecolor='white')
    axes[1].set_title('CLIENT_B — Cost per Purchase Distribution')
    axes[1].set_xlabel('Cost per Purchase ($)')
else:
    axes[1].text(0.5, 0.5, 'No ROAS or CPP data', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='grey')
    axes[1].set_title('CLIENT_B — ROAS / Cost per Purchase')

if 'purchases' in client_b.columns and client_b['purchases'].notna().sum() > 0:
    client_b_purch = client_b[client_b['purchases'].notna() & (client_b['purchases'] > 0)]
    axes[2].scatter(client_b_purch['spend'], client_b_purch['purchases'], alpha=0.7,
                    color=C_CLIENT_B, edgecolor='white', s=60)
    axes[2].set_title('CLIENT_B — Spend vs Purchases')
    axes[2].set_xlabel('Spend ($)')
    axes[2].set_ylabel('Purchases')
else:
    axes[2].text(0.5, 0.5, 'No purchase data', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=11, color='grey')
    axes[2].set_title('CLIENT_B — Spend vs Purchases')

plt.tight_layout()
plt.show()

### 4.5  Client E - Reach / Awareness Deep Dive

- **Goal:** Reach / Awareness (CPM)  
- **Industry:** Home improvement

In [ ]:
# ── 4.5  CLIENT_E — Reach / Awareness deep dive ─────────────────────────────
wl = cleaned['CLIENT_E'].copy()
C_WL = WB_PALETTE['CLIENT_E']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if 'cpm' in wl.columns and wl['cpm'].notna().sum() > 0:
    axes[0].hist(wl['cpm'].dropna(), bins=8, color=C_WL, edgecolor='white')
    axes[0].axvline(wl['cpm'].median(), color=WB_PALETTE['ref_line'], linestyle='--',
                    label=f"Median: ${wl['cpm'].median():.2f}")
    axes[0].set_title('CLIENT_E — CPM Distribution')
    axes[0].set_xlabel('CPM ($)')
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, 'No CPM data', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=11, color='grey')

if 'reach' in wl.columns and wl['reach'].notna().sum() > 0:
    axes[1].scatter(wl['spend'], wl['reach'], alpha=0.7, color=C_WL,
                    edgecolor='white', s=60)
    axes[1].set_title('CLIENT_E — Spend vs Reach')
    axes[1].set_xlabel('Spend ($)')
    axes[1].set_ylabel('Reach')
    axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
else:
    axes[1].text(0.5, 0.5, 'No reach data', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='grey')

if 'frequency' in wl.columns and wl['frequency'].notna().sum() > 0:
    axes[2].hist(wl['frequency'].dropna(), bins=8, color=C_WL, edgecolor='white')
    axes[2].axvline(3.0, color=WB_PALETTE['ref_line'], linestyle='--', label='Meta threshold (3.0)')
    axes[2].set_title('CLIENT_E — Frequency Distribution')
    axes[2].set_xlabel('Frequency')
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, 'No frequency data', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=11, color='grey')

plt.tight_layout()
plt.show()

print("CLIENT_E Summary Statistics:")
wl_desc_cols = [c for c in ['spend', 'impressions', 'reach', 'cpm', 'frequency']
                if c in wl.columns]
print(wl[wl_desc_cols].describe().round(2).to_string())

### 4.6  Client D - Lead Generation Overview

- **Goal:** Leads  
- **Industry:** Science / Health




In [ ]:
# ── 4.6  CLIENT_D — Lead generation overview ─────────────────────────────────
gk = cleaned['CLIENT_D'].copy()
C_GK = WB_PALETTE['CLIENT_D']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(gk['spend'].dropna(), bins=20, color=C_GK, edgecolor='white')
axes[0].axvline(gk['spend'].median(), color=WB_PALETTE['ref_line'], linestyle='--',
                label=f"Median: ${gk['spend'].median():.2f}")
axes[0].set_title('CLIENT_D — Spend Distribution')
axes[0].set_xlabel('Spend ($)')
axes[0].legend()

if 'reach' in gk.columns and gk['reach'].notna().sum() > 0:
    axes[1].scatter(gk['impressions'], gk['reach'], alpha=0.65, color=C_GK,
                    edgecolor='white', s=40)
    axes[1].set_title('CLIENT_D — Impressions vs Reach')
    axes[1].set_xlabel('Impressions')
    axes[1].set_ylabel('Reach')
else:
    axes[1].text(0.5, 0.5, 'No reach data', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='grey')

if 'all_leads' in gk.columns and (gk['all_leads'] > 0).any():
    gk_leads = gk[gk['all_leads'] > 0].copy()
    gk_leads['cpl_calc'] = gk_leads['spend'] / gk_leads['all_leads']
    monthly_gk = gk_leads.groupby(gk_leads['date_start'].dt.to_period('M'))['cpl_calc'].mean()
    monthly_gk.index = monthly_gk.index.to_timestamp()
    axes[2].plot(monthly_gk.index, monthly_gk.values, marker='o', color=C_GK, linewidth=2)
    axes[2].set_title('CLIENT_D — Avg CPL Over Time')
    axes[2].set_xlabel('Month')
    axes[2].set_ylabel('CPL ($)')
    axes[2].tick_params(axis='x', rotation=45)
else:
    axes[2].axis('off')
    gk_summary_cols = [c for c in ['spend', 'impressions', 'reach', 'cpm']
                       if c in gk.columns and gk[c].notna().sum() > 0]
    if len(gk_summary_cols) > 0:
        summary_text = "CLIENT_D Key Metrics\n" + "-" * 30
        for col in gk_summary_cols:
            val = gk[col].mean()
            if col in ['spend', 'cpm', 'cost_per_lead']:
                summary_text += f"\n  {col:<20} ${val:>10,.2f}"
            else:
                summary_text += f"\n  {col:<20} {val:>10,.0f}"
        axes[2].text(0.1, 0.5, summary_text, ha='left', va='center',
                     transform=axes[2].transAxes, fontsize=10, fontfamily='monospace')
    axes[2].set_title('CLIENT_D — Summary')

plt.tight_layout()
plt.show()

### 4.7  KPI Correlation Heatmap - Client A


In [ ]:
# ── 4.7  KPI correlation heatmap — CLIENT_A ──────────────────────────────────────
client_a_corr_cols = ['spend', 'impressions', 'reach', 'frequency', 'cpm',
                 'link_clicks', 'ctr_link', 'cpc_link', 'all_leads', 'cpl_calc']
client_a_corr_cols = [c for c in client_a_corr_cols if c in client_a.columns]

corr = client_a[client_a_corr_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=WB_CMAP_DIV,
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('CLIENT_A — KPI Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Cross-Client Feature Alignment

We identify the **common features** available across clients and separate the
goal-specific KPIs. This is important because the XGBoost models will
ultimately be trained per-goal-type (Leads vs Purchases vs Reach).

In [ ]:
# ── Universal features (present in all/most clients) ────────────────────────
UNIVERSAL_FEATURES = [
    'spend', 'impressions', 'reach', 'frequency',
    'cpm', 'ctr_link', 'cpc_link', 'clicks_all', 'ctr_all', 'cpc_all',
]

# Goal-specific target KPIs (CPL and CPT focus per March 13 agreement)
GOAL_KPIS = {
    'leads':     {'clients': ['CLIENT_A', 'CLIENT_D'], 'target': 'cpl_calc',          'label': 'Cost Per Lead (CPL)'},
    'purchases': {'clients': ['CLIENT_C', 'CLIENT_B'],   'target': 'cost_per_purchase',  'label': 'Cost Per Purchase (CPT)'},
    'reach':     {'clients': ['CLIENT_E'],     'target': 'cpm',                'label': 'CPM (Reach campaigns)'},
}

print("Feature availability by client:")
print(f"{'Column':<35}", end='')
for name in cleaned.keys():
    print(f"{name:>10}", end='')
print()
print('~' * 95)

all_cols = set()
for df in cleaned.values():
    all_cols.update(df.columns)

for col in sorted(all_cols):
    if col.startswith('_') or col in ['campaign_name', 'adset_name', 'ad_name', 'month',
                                        'delivery_status', 'delivery_level', 'result_type',
                                        'attribution_setting', 'date_start', 'date_end',
                                        'year_month']:
        continue
    is_critical = '*' if (CRITICAL_FEATURE_COLS and col in CRITICAL_FEATURE_COLS) else ' '
    row = f"{is_critical} {col:<33}"
    for name, df in cleaned.items():
        if col in df.columns:
            non_null = df[col].notna().sum()
            row += f"{'+ '+str(non_null):>10}"
        else:
            row += f"{'--':>10}"
    print(row)

if CRITICAL_FEATURE_COLS:
    print(f"\n  * = Critical KPI (from the client analyst's KPI guide)")

## 6. Goal-Specific KPI Analysis — CPL & CPT Focus

### Target metrics (agreed March 13 internal session):
- **CPL (Cost Per Lead)** — for Client A and Client D
- **CPT (Cost Per Purchase)** — for Client C and Client B

These metrics serve both as the classification target and as the threshold
control for the recommendation engine.

### Recommendation thresholds

| Recommendation | Meaning | Trigger logic |
|---------------|---------|---------------|
| **Scale Up** | Increase budget — strong efficiency | CPL/CPT significantly below benchmark |
| **Maintain** | Keep current settings | CPL/CPT near benchmark, stable |
| **Optimise** | Adjust spend allocation or placement | CPL/CPT above benchmark but improvable |
| **Pause** | Suspend campaign — poor return | CPL/CPT significantly above benchmark OR low volume |

> **THRESHOLD STATUS:**
> The thresholds below (70% / 110% / 160% of benchmark) are initial defaults.
> The client analyst and the client lead have not yet formally validated these. The team is
> proceeding with these values and will present them for feedback.


In [ ]:
# ── 6.1  Compute benchmarks ─────────────────────────────────────────────────

benchmarks = {}

# CLIENT_A — CPL benchmark (weighted average)
client_a_bench = cleaned['CLIENT_A'].copy()
client_a_bench['cpl_calc'] = client_a_bench['spend'] / client_a_bench['all_leads'].replace(0, np.nan)
client_a_valid = client_a_bench[client_a_bench['all_leads'] > 0]
weighted_cpl = client_a_valid['spend'].sum() / client_a_valid['all_leads'].sum()
simple_cpl   = client_a_valid['cpl_calc'].mean()
benchmarks['CLIENT_A_CPL_weighted'] = weighted_cpl
benchmarks['CLIENT_A_CPL_simple']   = simple_cpl
print(f"CLIENT_A — Weighted avg CPL: ${weighted_cpl:.2f}  |  Simple mean CPL: ${simple_cpl:.2f}")
print(f"  Use WEIGHTED avg (${weighted_cpl:.2f}) — simple mean is skewed by low-spend rows")

# CLIENT_C — CPT benchmark (Cost Per Purchase)
client_c_data = cleaned['CLIENT_C']
client_c_has_purchases = 'purchases' in client_c_data.columns and client_c_data['purchases'].notna().sum() > 0

if client_c_has_purchases:
    client_c_valid = client_c_data[client_c_data['purchases'] > 0].copy()
    weighted_cpt = client_c_valid['spend'].sum() / client_c_valid['purchases'].sum()
    median_roas  = client_c_valid['roas'].median() if 'roas' in client_c_valid.columns else None
    benchmarks['CLIENT_C_CPT_weighted'] = weighted_cpt
    if median_roas is not None:
        benchmarks['CLIENT_C_ROAS_median']  = median_roas
        print(f"\nCLIENT_C — Weighted avg CPT: ${weighted_cpt:.2f}  |  Median ROAS: {median_roas:.2f}x")
    else:
        print(f"\nCLIENT_C — Weighted avg CPT: ${weighted_cpt:.2f}  |  ROAS: not available")
else:
    client_c_valid = client_c_data.copy()
    print(f"\nCLIENT_C — No purchase columns ({len(client_c_valid)} rows).")
    print("       Cannot compute CPT benchmark.")
    if 'cpm' in client_c_valid.columns:
        client_c_cpm_median = client_c_valid['cpm'].median()
        benchmarks['CLIENT_C_CPM_median'] = client_c_cpm_median
        print(f"       Fallback: CLIENT_C Median CPM = ${client_c_cpm_median:.2f}")

# CLIENT_E — CPM benchmark
client_e_valid = cleaned['CLIENT_E'][cleaned['CLIENT_E']['cpm'].notna()]
median_cpm = client_e_valid['cpm'].median()
benchmarks['CLIENT_E_CPM_median'] = median_cpm
print(f"\nClient E — Median CPM: ${median_cpm:.2f}")

# CLIENT_D — CPL benchmark
client_d_bench = cleaned['CLIENT_D'].copy()
if 'all_leads' in client_d_bench.columns:
    client_d_bench['cpl_calc'] = client_d_bench['spend'] / client_d_bench['all_leads'].replace(0, np.nan)
    client_d_leads = client_d_bench[client_d_bench['all_leads'] > 0]
    if len(client_d_leads) > 5:
        client_d_weighted_cpl = client_d_leads['spend'].sum() / client_d_leads['all_leads'].sum()
        benchmarks['CLIENT_D_CPL_weighted'] = client_d_weighted_cpl
        print(f"\nCLIENT_D — Weighted avg CPL: ${client_d_weighted_cpl:.2f}  ({len(client_d_leads)} rows with leads)")
    else:
        print(f"\nCLIENT_D — Only {len(client_d_leads)} rows with leads. Benchmark may be unreliable.")

# CLIENT_A DAILY — CPL benchmark (separate from monthly — different granularity)
if 'client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0:
    client_a_daily_bench = client_a_daily_cleaned.copy()
    # Compute CPL for daily rows
    for leads_col in ['all_leads', 'leads']:
        if leads_col in client_a_daily_bench.columns:
            client_a_daily_bench['cpl_calc'] = client_a_daily_bench['spend'] / pd.to_numeric(
                client_a_daily_bench[leads_col], errors='coerce'
            ).replace(0, np.nan)
            client_a_daily_leads = client_a_daily_bench[
                pd.to_numeric(client_a_daily_bench[leads_col], errors='coerce') > 0
            ]
            if len(client_a_daily_leads) > 10:
                daily_weighted_cpl = (
                    client_a_daily_leads['spend'].sum() /
                    pd.to_numeric(client_a_daily_leads[leads_col], errors='coerce').sum()
                )
                benchmarks['CLIENT_A_DAILY_CPL_weighted'] = daily_weighted_cpl
                print(f"\nCLIENT_A DAILY — Weighted avg CPL: ${daily_weighted_cpl:.2f}  "
                      f"({len(client_a_daily_leads)} rows with leads out of {len(client_a_daily_cleaned)})")
                print(f"  NOTE: Daily CPL benchmark differs from monthly (${weighted_cpl:.2f}) "
                      f"due to granularity difference.")
            else:
                print(f"\nCLIENT_A DAILY — Only {len(client_a_daily_leads)} rows with leads. "
                      f"Insufficient for separate benchmark.")
            break

print("\nBenchmarks computed")


Note: CPT is Cost per Thousand (the cost an advertiser pays for every impressions (views) of an ad)

In [ ]:
# ── 6.2  Recommendation assignment function ──────────────────────────────────

def assign_recommendation(cost_metric, benchmark):
    """
    Assign recommendation based on cost metric vs benchmark ratio.

    Uses the sigmoid-inspired threshold framework agreed on March 13:
      Scale Up  : cost < 70% of benchmark   -> strong efficiency
      Maintain  : 70-110% of benchmark       -> on-target
      Optimise  : 110-160% of benchmark      -> above benchmark but recoverable
      Pause     : cost > 160% of benchmark   -> poor return
    """
    if pd.isna(cost_metric) or pd.isna(benchmark) or benchmark == 0:
        return np.nan
    ratio = cost_metric / benchmark
    if ratio < 0.70:
        return 'Scale Up'
    elif ratio < 1.10:
        return 'Maintain'
    elif ratio < 1.60:
        return 'Optimise'
    else:
        return 'Pause'


def assign_binary_label(cost_metric, benchmark):
    """Binary simplification: Scale vs Do Not Scale."""
    if pd.isna(cost_metric) or pd.isna(benchmark) or benchmark == 0:
        return np.nan
    return 'Scale' if cost_metric / benchmark < 1.10 else 'Do Not Scale'

In [ ]:
# ── 6.3  Apply labels to CLIENT_A (CPL) ──────────────────────────────────────────
client_a_label_cols = [c for c in ['campaign_name', 'adset_name', 'ad_name',
                               'date_start', 'year_month',
                               'spend', 'impressions', 'reach', 'frequency', 'cpm',
                               'link_clicks', 'ctr_link', 'cpc_link',
                               'all_leads', 'cpl_calc'] if c in client_a_valid.columns]
client_a_labelled = client_a_valid[client_a_label_cols].copy()

if 'date_end' in client_a_valid.columns:
    client_a_labelled['date_end'] = client_a_valid['date_end'].values

B_client_a = benchmarks['CLIENT_A_CPL_weighted']

client_a_labelled['recommendation'] = client_a_labelled['cpl_calc'].apply(
    lambda x: assign_recommendation(x, B_client_a)
)
client_a_labelled['recommendation_binary'] = client_a_labelled['cpl_calc'].apply(
    lambda x: assign_binary_label(x, B_client_a)
)

counts = client_a_labelled['recommendation'].value_counts()
print(f"CLIENT_A Recommendation distribution (benchmark CPL = ${B_client_a:.2f}):")
print(counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rec_colors = [WB_PALETTE.get(c, '#999') for c in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=rec_colors, edgecolor='white')
axes[0].bar_label(bars)
axes[0].set_title(f'CLIENT_A — 4-Class Distribution\n(Benchmark CPL = ${B_client_a:.2f})', fontweight='bold')
axes[0].set_ylabel('Count')

bin_counts = client_a_labelled['recommendation_binary'].value_counts()
bin_colors = [WB_PALETTE['Scale Up'] if c == 'Scale' else WB_PALETTE['Pause'] for c in bin_counts.index]
bars2 = axes[1].bar(bin_counts.index, bin_counts.values, color=bin_colors, edgecolor='white')
axes[1].bar_label(bars2)
axes[1].set_title(f'CLIENT_A — Binary Distribution\n(Benchmark CPL = ${B_client_a:.2f})', fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.4  Apply labels to CLIENT_C (CPT) ────────────────────────────────────────
desired_client_c_cols = [
    'campaign_name', 'adset_name', 'ad_name',
    'date_start', 'year_month',
    'spend', 'impressions', 'reach', 'frequency', 'cpm',
    'link_clicks', 'ctr_link', 'cpc_link',
    'purchases', 'cost_per_purchase', 'roas'
]
available_client_c_cols = [col for col in desired_client_c_cols if col in client_c_valid.columns]
client_c_labelled = client_c_valid[available_client_c_cols].copy()

if 'date_end' in client_c_valid.columns:
    client_c_labelled['date_end'] = client_c_valid['date_end'].values

if 'CLIENT_C_CPT_weighted' in benchmarks and 'cost_per_purchase' in client_c_labelled.columns:
    B_client_c = benchmarks['CLIENT_C_CPT_weighted']
    client_c_labelled['recommendation'] = client_c_labelled['cost_per_purchase'].apply(
        lambda x: assign_recommendation(x, B_client_c)
    )
    client_c_labelled['recommendation_binary'] = client_c_labelled['cost_per_purchase'].apply(
        lambda x: assign_binary_label(x, B_client_c)
    )
else:
    B_client_c = None
    print("CLIENT_C: No CPT benchmark or cost_per_purchase data — cannot assign labels.")
    client_c_labelled['recommendation'] = np.nan
    client_c_labelled['recommendation_binary'] = np.nan

if 'recommendation' in client_c_labelled.columns and client_c_labelled['recommendation'].notna().any():
    counts_client_c = client_c_labelled['recommendation'].value_counts()
    print(f"CLIENT_C Recommendation distribution (benchmark CPT = ${B_client_c:.2f}):")
    print(counts_client_c.to_string())

    fig, ax = plt.subplots(figsize=(7, 4))
    rec_colors_client_c = [WB_PALETTE.get(c, '#999') for c in counts_client_c.index]
    bars = ax.bar(counts_client_c.index, counts_client_c.values, color=rec_colors_client_c, edgecolor='white')
    ax.bar_label(bars)
    ax.set_title(f'CLIENT_C — Recommendation Distribution\n(Benchmark CPT = ${B_client_c:.2f})', fontweight='bold')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("CLIENT_C: No recommendation labels to plot (purchase data missing).")

## 7. Recommendation Threshold Framework

### Optimise sub-signals (from the client analyst's brief)

When `Optimise` is triggered, the engine surfaces *which lever to pull*.
The Optimise category focuses on **spend allocation and placement control**
rather than audience targeting, because Meta's platform AI overrides manual
audience settings.

In [ ]:
# ── 7.1  Optimise sub-signals ────────────────────────────────────────────────

def add_optimise_signals(df, goal='leads'):
    """
    Add sub-signals to support the Optimise recommendation.

    Signals aligned with the agency partner's agreed optimisation levers:
      1. Spend pacing    — daily spend relative to peers (budget allocation)
      2. CPM efficiency  — cost-per-impression as proxy for placement quality
      3. Frequency fatigue — high frequency suggests creative refresh needed
    """
    df = df.copy()

    # Signal 1: Spend pacing
    if 'date_end' in df.columns and df['date_end'].notna().any():
        df['days_active'] = (df['date_end'] - df['date_start']).dt.days + 1
    else:
        df['days_active'] = np.nan
    df['daily_spend'] = df['spend'] / df['days_active'].replace(0, np.nan)

    q25_daily = df['daily_spend'].quantile(0.25)
    df['flag_low_daily_spend'] = (df['daily_spend'] < q25_daily).astype(int)

    # Signal 2: CPM efficiency (placement quality proxy)
    if 'cpm' in df.columns:
        median_cpm = df['cpm'].median()
        df['flag_high_cpm'] = (df['cpm'] > median_cpm * 1.3).astype(int)
    else:
        df['flag_high_cpm'] = 0

    # Signal 3: Frequency fatigue
    if 'frequency' in df.columns:
        df['flag_high_frequency'] = (df['frequency'] > 3.0).astype(int)
    else:
        df['flag_high_frequency'] = 0

    # Summary signal for Optimise rows
    df['optimise_focus'] = np.where(
        df['recommendation'] == 'Optimise',
        df[['flag_low_daily_spend', 'flag_high_cpm', 'flag_high_frequency']].apply(
            lambda row: ', '.join(filter(None, [
                'Budget pacing/spend allocation' if row['flag_low_daily_spend'] else '',
                'Placement cost (high CPM)'      if row['flag_high_cpm'] else '',
                'Creative refresh (frequency)'   if row['flag_high_frequency'] else '',
            ])) or 'Review overall efficiency',
            axis=1
        ),
        ''
    )

    return df

client_a_labelled = add_optimise_signals(client_a_labelled, goal='leads')

# ── CLIENT_A DAILY — Label and prepare for modelling ──────────────────────────
# This promotes CLIENT_A daily from EDA-only to a trainable dataset.
# Uses its own benchmark (daily granularity) rather than the monthly one.

client_a_daily_labelled = pd.DataFrame()
CLIENT_A_DAILY_MODEL_READY = False

if ('client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0
        and 'CLIENT_A_DAILY_CPL_weighted' in benchmarks):

    client_a_d = client_a_daily_cleaned.copy()

    # Identify the leads column
    leads_col_d = None
    for client_b in ['all_leads', 'leads']:
        if client_b in client_a_d.columns:
            leads_col_d = client_b
            break

    if leads_col_d is not None:
        client_a_d[leads_col_d] = pd.to_numeric(client_a_d[leads_col_d], errors='coerce')
        client_a_d['cpl_calc'] = client_a_d['spend'] / client_a_d[leads_col_d].replace(0, np.nan)
        client_a_d_valid = client_a_d[client_a_d[leads_col_d] > 0].copy()

        B_client_a_daily = benchmarks['CLIENT_A_DAILY_CPL_weighted']

        # Select columns for labelling (defensive — only pick what exists)
        desired_cols = ['campaign_name', 'adset_name', 'ad_name',
                        'date_start', 'year_month', 'day_of_week_num',
                        'spend', 'impressions', 'reach', 'frequency', 'cpm',
                        'link_clicks', 'ctr_link', 'cpc_link',
                        leads_col_d, 'cpl_calc']
        available_cols = [c for c in desired_cols if c in client_a_d_valid.columns]
        client_a_daily_labelled = client_a_d_valid[available_cols].copy()

        # Rename leads column to all_leads for consistency
        if leads_col_d != 'all_leads' and leads_col_d in client_a_daily_labelled.columns:
            client_a_daily_labelled = client_a_daily_labelled.rename(columns={leads_col_d: 'all_leads'})

        if 'date_end' in client_a_d_valid.columns:
            client_a_daily_labelled['date_end'] = client_a_d_valid['date_end'].values

        # Apply labels using DAILY benchmark
        client_a_daily_labelled['recommendation'] = client_a_daily_labelled['cpl_calc'].apply(
            lambda x: assign_recommendation(x, B_client_a_daily)
        )
        client_a_daily_labelled['recommendation_binary'] = client_a_daily_labelled['cpl_calc'].apply(
            lambda x: assign_binary_label(x, B_client_a_daily)
        )

        # Tag granularity and source
        client_a_daily_labelled['_client'] = 'CLIENT_A'
        client_a_daily_labelled['_granularity'] = 'daily'

        # Add optimise sub-signals
        client_a_daily_labelled = add_optimise_signals(client_a_daily_labelled, goal='leads')

        CLIENT_A_DAILY_MODEL_READY = True

        counts_d = client_a_daily_labelled['recommendation'].value_counts()
        print(f"\nCLIENT_A DAILY Recommendation distribution "
              f"(benchmark CPL = ${B_client_a_daily:.2f}, {len(client_a_daily_labelled)} rows):")
        print(counts_d.to_string())
        print(f"\nCLIENT_A DAILY Binary distribution:")
        print(client_a_daily_labelled['recommendation_binary'].value_counts().to_string())

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        rec_colors_d = [WB_PALETTE.get(c, '#999') for c in counts_d.index]
        bars = axes[0].bar(counts_d.index, counts_d.values, color=rec_colors_d, edgecolor='white')
        axes[0].bar_label(bars)
        axes[0].set_title(f'CLIENT_A DAILY — 4-Class Distribution\n'
                          f'(Benchmark CPL = ${B_client_a_daily:.2f}, n={len(client_a_daily_labelled)})',
                          fontweight='bold')
        axes[0].set_ylabel('Count')

        bin_d = client_a_daily_labelled['recommendation_binary'].value_counts()
        bin_colors_d = [WB_PALETTE['Scale Up'] if c == 'Scale' else WB_PALETTE['Pause']
                        for c in bin_d.index]
        bars2 = axes[1].bar(bin_d.index, bin_d.values, color=bin_colors_d, edgecolor='white')
        axes[1].bar_label(bars2)
        axes[1].set_title(f'CLIENT_A DAILY — Binary Distribution\n'
                          f'(Benchmark CPL = ${B_client_a_daily:.2f})',
                          fontweight='bold')
        axes[1].set_ylabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print("\nCLIENT_A DAILY — No leads column found. Cannot label.")
else:
    print("\nCLIENT_A DAILY — Not available or no benchmark computed.")

if client_c_labelled['recommendation'].notna().any():
    client_c_labelled = add_optimise_signals(client_c_labelled, goal='purchases')
else:
    print("CLIENT_C: Skipping optimise signals (no recommendations to enrich).")

# Preview Optimise rows
print("CLIENT_A — Sample Optimise rows with sub-signals:")
optimise_sample = client_a_labelled[client_a_labelled['recommendation'] == 'Optimise'][
    ['campaign_name', 'year_month', 'cpl_calc', 'daily_spend', 'cpm', 'frequency', 'optimise_focus']
].head(5)
print(optimise_sample.to_string(index=False))

## 8. Feature Engineering for XGBoost

Features are filtered by the KPI guide (critical metrics only) and include
derived ratio features for the classifier.

In [ ]:
# ── 8.1  Feature engineering pipeline ────────────────────────────────────────

# NOTE: CLIENT_A daily CSV exports do not include 'cpm' or 'ctr_link' columns.
# These are present in the monthly xlsx but absent from the daily Meta export.
# The daily feature matrix therefore has 9 features vs 12 for monthly.
# This is a data availability constraint, not a pipeline error.
# Impact: cpm and ctr_link cannot be used as predictors in the daily model.

def engineer_features(df, target_col, critical_cols=None):
    """
    Build the feature matrix X and label vector y for the XGBoost model.

    Parameters
    ----------
    df : pd.DataFrame
        Labelled dataset (must contain 'recommendation' column).
    target_col : str
        The goal-specific KPI column (e.g. 'cpl_calc', 'cost_per_purchase').
    critical_cols : list or None
        If provided (from KPI guide), only these columns are used as core
        features.

    Features:
    - Core spend/efficiency metrics (filtered by KPI guide if available)
    - Temporal features (month-of-year, quarter, day-of-week for daily)
    - Rolling averages (7-day for daily, 3-month for monthly)
    - Ratio features (spend efficiency ratios)
    """
    df = df.copy().sort_values('date_start')
    granularity = df['_granularity'].iloc[0] if '_granularity' in df.columns else 'monthly'

    # ── Core features (KPI-guide-aware) ────────────────────────────────────
    all_candidates = ['spend', 'impressions', 'reach', 'frequency', 'cpm',
                      'ctr_link', 'cpc_link', 'clicks_all', 'ctr_all']

    if critical_cols is not None and len(critical_cols) > 0:
        core_candidates = [c for c in all_candidates if c in critical_cols]
        if 'spend' not in core_candidates and 'spend' in df.columns:
            core_candidates.insert(0, 'spend')
        dropped = set(all_candidates) - set(core_candidates) - {'spend'}
        if dropped:
            print(f"  [KPI filter] Excluded non-critical features: {sorted(dropped)}")
    else:
        core_candidates = all_candidates

    feature_cols = [c for c in core_candidates if c in df.columns]

    # ── Temporal features ──────────────────────────────────────────────────
    if 'date_start' in df.columns:
        df['month_of_year'] = df['date_start'].dt.month
        df['quarter']       = df['date_start'].dt.quarter
        feature_cols += ['month_of_year', 'quarter']

    # Day-of-week feature (daily data only — useful for XGBoost)
    if granularity == 'daily' and 'day_of_week_num' in df.columns:
        feature_cols.append('day_of_week_num')
    elif granularity == 'daily' and 'date_start' in df.columns:
        df['day_of_week_num'] = df['date_start'].dt.dayofweek
        feature_cols.append('day_of_week_num')

    # ── Ratio features ─────────────────────────────────────────────────────
    df['ctr_x_spend']   = df.get('ctr_link', 0) * df['spend']
    df['spend_per_imp'] = df['spend'] / df['impressions'].replace(0, np.nan)
    df['reach_ratio']   = df['reach'] / df['impressions'].replace(0, np.nan)
    feature_cols += ['ctr_x_spend', 'spend_per_imp', 'reach_ratio']

    # ── Rolling averages (granularity-aware) ──────────────────────────────
    if 'date_start' in df.columns and len(df) >= 6:
        if granularity == 'daily':
            # 7-day rolling average per campaign (avoid mixing time series)
            group_col = 'adset_name' if 'adset_name' in df.columns else 'campaign_name'
            if group_col in df.columns:
                df = df.sort_values([group_col, 'date_start'])
                for col in [target_col, 'cpm', 'ctr_link']:
                    if col in df.columns:
                        new_col = f'{col}_7d_avg'
                        df[new_col] = df.groupby(group_col)[col].transform(
                            lambda x: x.rolling(7, min_periods=1).mean()
                        )
                        feature_cols.append(new_col)
            else:
                # Fallback: global rolling
                df = df.set_index('date_start').sort_index()
                for col in [target_col, 'cpm', 'ctr_link']:
                    if col in df.columns:
                        new_col = f'{col}_7d_avg'
                        df[new_col] = df[col].rolling(7, min_periods=1).mean()
                        feature_cols.append(new_col)
                df = df.reset_index()
        else:
            # Monthly: 3-month trailing average
            df = df.set_index('date_start').sort_index()
            for col in [target_col, 'cpm', 'ctr_link']:
                if col in df.columns:
                    new_col = f'{col}_3m_avg'
                    df[new_col] = df[col].rolling(3, min_periods=1).mean()
                    feature_cols.append(new_col)
            df = df.reset_index()

    # ── Build final feature matrix ─────────────────────────────────────────
    feature_cols = list(dict.fromkeys(feature_cols))  # deduplicate
    feature_cols = [c for c in feature_cols if c in df.columns]

    X = df[feature_cols].copy()
    y = df['recommendation'].copy()
    y_binary = df['recommendation_binary'].copy() if 'recommendation_binary' in df.columns else None

    # Handle nulls — fill with median
    X = X.fillna(X.median(numeric_only=True))

    print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
    print(f"Features: {feature_cols}")
    print(f"Label distribution (4-class):\n{y.value_counts().to_string()}")
    if y_binary is not None:
        print(f"Label distribution (binary):\n{y_binary.value_counts().to_string()}")

    return X, y, y_binary, feature_cols, df


print("=== CLIENT_A Feature Engineering ===")
X_client_a, y_client_a, y_client_a_bin, client_a_features, client_a_labelled = engineer_features(
    client_a_labelled, target_col='cpl_calc', critical_cols=CRITICAL_FEATURE_COLS
)

# ── CLIENT_A DAILY Feature Engineering ─────────────────────────────────────────
X_client_a_daily = pd.DataFrame()
y_client_a_daily = pd.Series(dtype=str)
y_client_a_daily_bin = None
client_a_daily_features = []

if CLIENT_A_DAILY_MODEL_READY and len(client_a_daily_labelled) > 0:
    print("\n=== CLIENT_A DAILY Feature Engineering ===")
    (X_client_a_daily, y_client_a_daily, y_client_a_daily_bin,
     client_a_daily_features, client_a_daily_labelled) = engineer_features(
        client_a_daily_labelled, target_col='cpl_calc', critical_cols=CRITICAL_FEATURE_COLS
    )
else:
    print("\n=== CLIENT_A DAILY Feature Engineering: SKIPPED ===")

print("\n=== CLIENT_C Feature Engineering ===")
if client_c_labelled['recommendation'].notna().any():
    client_c_target = 'cost_per_purchase' if 'cost_per_purchase' in client_c_labelled.columns else 'cpm'
    X_client_c, y_client_c, y_client_c_bin, client_c_features, client_c_labelled = engineer_features(
        client_c_labelled, target_col=client_c_target, critical_cols=CRITICAL_FEATURE_COLS
    )
    CLIENT_C_MODEL_READY = True
else:
    print("  CLIENT_C has no recommendation labels — skipping feature engineering.")
    X_client_c = pd.DataFrame()
    y_client_c = pd.Series(dtype=str)
    y_client_c_bin = None
    client_c_features = []
    CLIENT_C_MODEL_READY = False

In [ ]:
# ── 8.2  Feature distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

plot_features = [c for c in client_a_features if c in X_client_a.columns][:8]
for i, feat in enumerate(plot_features):
    axes[i].hist(X_client_a[feat].dropna(), bins=20, color=WB_PALETTE['primary'],
                 alpha=0.65, edgecolor='white')
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('CLIENT_A — Feature Distributions (pre-normalisation)', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Data Export for Modelling

Export cleaned, labelled, and feature-engineered datasets ready for the
XGBoost training in Section 11 (and for reproducibility).

In [ ]:
# ── 9.1  Export processed datasets ──────────────────────────────────────────

OUTPUT_DIR = Path('/content/drive/MyDrive/<project-folder>
OUTPUT_DIR.mkdir(exist_ok=True)

# Export labelled datasets
client_a_labelled.to_csv(OUTPUT_DIR / 'client_a_labelled.csv', index=False)
client_c_labelled.to_csv(OUTPUT_DIR / 'client_c_labelled.csv', index=False)

# Export feature matrices
X_client_a_export = X_client_a.copy()
X_client_a_export['recommendation'] = y_client_a.values
X_client_a_export['recommendation_binary'] = y_client_a_bin.values if y_client_a_bin is not None else np.nan
X_client_a_export.to_csv(OUTPUT_DIR / 'client_a_features_X.csv', index=False)

if CLIENT_C_MODEL_READY and len(X_client_c) > 0:
    X_client_c_export = X_client_c.copy()
    X_client_c_export['recommendation'] = y_client_c.values
    X_client_c_export['recommendation_binary'] = y_client_c_bin.values if y_client_c_bin is not None else np.nan
    X_client_c_export.to_csv(OUTPUT_DIR / 'client_c_features_X.csv', index=False)
else:
    print("  CLIENT_C feature matrix not exported (no purchase data).")

# Export benchmarks
with open(OUTPUT_DIR / 'benchmarks.json', 'w') as f:
    json.dump(benchmarks, f, indent=2, default=str)

# Export KPI mapping
kpi_map_export = {
    'critical_feature_cols': CRITICAL_FEATURE_COLS,
    'goal_kpis': {k: {'target': v['target'], 'label': v['label']} for k, v in GOAL_KPIS.items()},
    'kpi_guide_available': KPI_GUIDE_AVAILABLE,
}
with open(OUTPUT_DIR / 'kpi_mapping.json', 'w') as f:
    json.dump(kpi_map_export, f, indent=2)

print("Exports complete:")
for f in OUTPUT_DIR.iterdir():
    print(f"   {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 10. Data Quality Summary & Cross-Client EDA


In [ ]:
# ── 10.1  Build the Windbreaker Master Dataset ─────────────────────────────
CLIENT_META = {
    'CLIENT_A':      {'client_name': 'Client A', 'goal': 'leads'},
    'CLIENT_C':     {'client_name': "Client C",         'goal': 'purchases'},
    'CLIENT_B':       {'client_name': 'Client B',  'goal': 'purchases'},
    'CLIENT_E': {'client_name': 'client_e',             'goal': 'reach'},
    'CLIENT_D':    {'client_name': 'CLIENT_D',               'goal': 'leads'},
}

master_frames = []
for name, df in cleaned.items():
    temp = df.copy()
    temp['client_name'] = CLIENT_META[name]['client_name']
    temp['client_goal'] = CLIENT_META[name]['goal']
    master_frames.append(temp)

windbreaker_master = pd.concat(master_frames, ignore_index=True)
print(f"Windbreaker Master: {len(windbreaker_master)} rows x {len(windbreaker_master.columns)} cols")
print(f"Clients: {windbreaker_master['client_name'].value_counts().to_string()}")

In [ ]:
# ── 10.2  Missing values per client ────────────────────────────────────────
print("\n" + "=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

for name, df in cleaned.items():
    print(f"\n{name} Missing Values (top 10):")
    miss = pd.DataFrame({
        'column': df.columns,
        'missing': df.isnull().sum().values,
        'missing_pct': (df.isnull().sum().values / len(df) * 100).round(2)
    })
    miss_filtered = miss[miss['missing'] > 0].sort_values('missing', ascending=False).head(10)
    if len(miss_filtered) == 0:
        print("  No missing values")
    else:
        print(miss_filtered.to_string(index=False))

In [ ]:
# ── 10.3  Key performance metrics per client ────────────────────────────────
print("\n" + "=" * 80)
print("KEY PERFORMANCE METRICS")
print("=" * 80)

for name, df in cleaned.items():
    meta = CLIENT_META[name]
    print(f"\n{name} ({meta['client_name']}) — Goal: {meta['goal']}")
    print(f"  Total rows: {len(df)}")
    print(f"  Total spend: ${df['spend'].sum():,.2f}")

    if meta['goal'] == 'leads':
        if 'all_leads' in df.columns and df['all_leads'].notna().any():
            print(f"  Total leads: {df['all_leads'].sum():.0f}")
        if 'cost_per_lead' in df.columns and df['cost_per_lead'].notna().any():
            print(f"  Avg CPL: ${df['cost_per_lead'].mean():.2f}")
        if 'ctr_link' in df.columns and df['ctr_link'].notna().any():
            print(f"  Avg CTR: {df['ctr_link'].mean():.2%}")
    elif meta['goal'] == 'purchases':
        if 'purchases' in df.columns and df['purchases'].notna().any():
            print(f"  Total purchases: {df['purchases'].sum():.0f}")
        if 'purchase_value' in df.columns and df['purchase_value'].notna().any():
            print(f"  Total revenue: ${df['purchase_value'].sum():,.2f}")
        if 'roas' in df.columns and df['roas'].notna().any():
            print(f"  Avg ROAS: {df['roas'].mean():.2f}")
        if 'cost_per_purchase' in df.columns and df['cost_per_purchase'].notna().any():
            print(f"  Avg CPT: ${df['cost_per_purchase'].mean():.2f}")
    elif meta['goal'] == 'reach':
        if 'reach' in df.columns and df['reach'].notna().any():
            print(f"  Total reach: {df['reach'].sum():,.0f}")
        if 'cpm' in df.columns and df['cpm'].notna().any():
            print(f"  Avg CPM: ${df['cpm'].mean():.2f}")

### Distribution of Campaign Spend



In [ ]:
# ── Cross-client charts ─────────────────────────────────────────────────────
plt.figure(figsize=(8,5))
windbreaker_master['spend'].hist(bins=50, color=WB_PALETTE['primary'], edgecolor='white')
plt.title("Distribution of Campaign Spend (All Clients)")
plt.xlabel("Spend (AUD)")
plt.ylabel("Frequency")
plt.show()

### Advertising Spend by Client


In [ ]:
plt.figure(figsize=(8,5))
spend_by_client = windbreaker_master.groupby('client_name')['spend'].sum().sort_values()
client_bar_colors = [WB_PALETTE.get(
    [k for k, v in CLIENT_META.items() if v['client_name'] == name][0], WB_PALETTE['primary'])
    for name in spend_by_client.index]
spend_by_client.plot(kind='barh', color=client_bar_colors, edgecolor='white')
plt.title("Total Advertising Spend by Client")
plt.xlabel("Spend (AUD)")
plt.ylabel("")
plt.tight_layout()
plt.show()


### Correlation Between Key Advertising Metrics

In [ ]:
# Correlation heatmap
corr_cols = [c for c in ['spend', 'ctr_link', 'cpc_link', 'cpm', 'cost_per_lead']
             if c in windbreaker_master.columns]
corr_data = windbreaker_master[corr_cols].dropna()

if len(corr_data) > 5:
    metrics_corr = corr_data.corr()
    plt.figure(figsize=(6,5))
    sns.heatmap(metrics_corr, annot=True, cmap=WB_CMAP, fmt='.2f')
    plt.title("Correlation Between Key Advertising Metrics")
    plt.tight_layout()
    plt.show()

### Relationship Between CTR and Cost per Lead (CPL)

In [ ]:
# CTR vs CPL
ctr_cpl_data = windbreaker_master[
    windbreaker_master['ctr_link'].notna() & windbreaker_master['cost_per_lead'].notna()
]

if len(ctr_cpl_data) > 0:
    plt.figure(figsize=(8,5))
    sns.scatterplot(
        data=ctr_cpl_data, x='ctr_link', y='cost_per_lead',
        hue='client_name',
        palette={v['client_name']: WB_PALETTE[k] for k, v in CLIENT_META.items()},
        alpha=0.65
    )
    plt.title("CTR vs Cost per Lead")
    plt.xlabel("CTR (Link)")
    plt.ylabel("CPL (AUD)")
    plt.tight_layout()
    plt.show()

### Campaign Efficiency Zones

In [ ]:
if len(ctr_cpl_data) > 0:
    plt.figure(figsize=(8,5))
    sns.scatterplot(data=ctr_cpl_data, x='ctr_link', y='cost_per_lead',
                    color=WB_PALETTE['secondary'], alpha=0.65)
    plt.axhline(ctr_cpl_data['cost_per_lead'].median(), linestyle='--', color='grey', alpha=0.65,
                label=f"Median CPL: ${ctr_cpl_data['cost_per_lead'].median():.0f}")
    plt.axvline(ctr_cpl_data['ctr_link'].median(), linestyle='--', color='grey', alpha=0.65,
                label=f"Median CTR: {ctr_cpl_data['ctr_link'].median():.2%}")
    plt.legend()
    plt.title("Campaign Efficiency Map")
    plt.xlabel("CTR (Link)")
    plt.ylabel("CPL (AUD)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  PHASE 2 — XGBoost MODEL TRAINING & EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

## 11. Phase 2 — XGBoost Model Training

### Model approach (agreed 13 of March during team working session)

The mentors strongly advised against a neural network for this dataset:
- **Small data:** Client A ~100 rows, Client C ~500 rows. Neural networks need
  thousands of rows to generalise well.
- **Interpretability:** XGBoost provides feature importance natively, which
  is a key deliverable for the agency partner.
- **Robustness:** Tree-based models handle mixed feature types, missing data,
  and small samples better than neural networks.

### v8 — Binary-to-4-class sigmoid mapping (the advisor's framework)

**PRIMARY approach (advisor guidance, March 13 — confirmed in Week 9):**
> Treat the final sigmoid output of a binary model as the probability of
> scaling up the campaign, then map that probability to four recommendation
> tiers: a high probability is a clear scale-up, a middle band means maintain,
> and lower probabilities fall to the remaining tiers.

The Week 9 review confirmed the recommendation to revert to a binary
classification approach and then use the resulting probabilities as
thresholds to create four distinct categories. Rationale: a small dataset
makes training a four-category model directly problematic.

Implementation:
1. Train a BINARY XGBoost classifier (Scale / Do Not Scale)
2. Extract the sigmoid probability P(Scale) for each row
3. Map P(Scale) to 4 recommendation tiers using thresholds:
   - P(Scale) > 0.70  ->  Scale Up     (high probability of scaling)
   - P(Scale) 0.40-0.70 -> Maintain    (moderate, hold steady)
   - P(Scale) 0.20-0.40 -> Optimise    (below midpoint, needs work)
   - P(Scale) < 0.20  ->  Pause        (low probability, poor return)
4. The probability value itself serves as the confidence score.

**SECONDARY approach (comparison):**
Direct 4-class XGBoost with softprob is retained for cross-validation
comparison only. It is NOT the primary deliverable.

| Model | Target | Client | Purpose |
|-------|--------|--------|---------|
| **Binary XGBoost (sigmoid)** | Scale / Do Not Scale | Client A, Client C | **PRIMARY** — sigmoid P(Scale) mapped to 4 tiers |
| **4-class XGBoost (softprob)** | Scale Up / Maintain / Optimise / Pause | Client A, Client C | SECONDARY — comparison only |

In [ ]:
if not XGBOOST_AVAILABLE:
    print("=" * 60)
    print("XGBOOST NOT INSTALLED — skipping Phase 2")
    print("Run: !pip install xgboost")
    print("=" * 60)
else:
    # ── 11.1  Helper: train XGBoost with probability outputs ───────────
    def train_xgboost_kfold(X, y, model_name='Model', n_splits=5, is_binary=False):
        """
        Train XGBoost using Stratified K-Fold CV and return the model,
        predictions, probabilities, and evaluation metrics.

        v8: Binary model is PRIMARY (sigmoid output). 4-class retained
        for comparison only. F1 macro used per advisor agreement.
        """
        # Drop any rows with NaN labels
        valid_mask = y.notna()
        X_clean = X[valid_mask].copy()
        y_clean = y[valid_mask].copy()

        # Encode labels
        le = LabelEncoder()
        y_encoded = le.fit_transform(y_clean)
        class_names = le.classes_

        # Scale features
        scaler = StandardScaler()
        X_scaled = pd.DataFrame(
            scaler.fit_transform(X_clean),
            columns=X_clean.columns,
            index=X_clean.index
        )

        # Adjust n_splits if any class has fewer samples
        min_class_count = pd.Series(y_encoded).value_counts().min()
        actual_splits = min(n_splits, min_class_count)
        if actual_splits < 2:
            print(f"  WARNING: A class has only {min_class_count} sample(s).")
            print(f"  Cannot run K-Fold CV. Training on full dataset instead.")
            actual_splits = None

        # Configure XGBoost — v8: binary is primary (sigmoid); softprob for comparison
        if is_binary:
            xgb_params = {
                'objective': 'binary:logistic',
                'eval_metric': 'logloss',
                'max_depth': 4,
                'learning_rate': 0.1,
                'n_estimators': 100,
                'use_label_encoder': False,
                'random_state': 42,
                'verbosity': 0,
            }
        else:
            num_classes = len(class_names)
            xgb_params = {
                'objective': 'multi:softprob',  # v8: comparison only
                'num_class': num_classes,
                'eval_metric': 'mlogloss',
                'max_depth': 4,
                'learning_rate': 0.1,
                'n_estimators': 100,
                'use_label_encoder': False,
                'random_state': 42,
                'verbosity': 0,
            }

        model = xgb.XGBClassifier(**xgb_params)

        if actual_splits and actual_splits >= 2:
            skf = StratifiedKFold(n_splits=actual_splits, shuffle=True, random_state=42)
            y_pred_encoded = cross_val_predict(model, X_scaled, y_encoded, cv=skf)
            y_pred = le.inverse_transform(y_pred_encoded)

            # Train on full dataset for feature importance and probability extraction
            model.fit(X_scaled, y_encoded)

            # Extract probabilities from the full-dataset model
            y_proba = model.predict_proba(X_scaled)

            # Metrics
            acc = accuracy_score(y_clean, y_pred)
            f1 = f1_score(y_encoded, y_pred_encoded, average='macro')
            report = classification_report(y_clean, y_pred, zero_division=0)

            print(f"\n{'=' * 60}")
            print(f"{model_name} — {actual_splits}-Fold Cross-Validation Results")
            print(f"{'=' * 60}")
            print(f"  Accuracy:    {acc:.3f}")
            print(f"  Macro F1:    {f1:.3f}")
            print(f"\n{report}")
        else:
            model.fit(X_scaled, y_encoded)
            y_pred_encoded = model.predict(X_scaled)
            y_pred = le.inverse_transform(y_pred_encoded)
            y_proba = model.predict_proba(X_scaled)
            acc = accuracy_score(y_clean, y_pred)
            report = classification_report(y_clean, y_pred, zero_division=0)

            print(f"\n{'=' * 60}")
            print(f"{model_name} — Full Dataset (no CV, too few samples)")
            print(f"{'=' * 60}")
            print(f"  Training Accuracy: {acc:.3f}  (WARNING: likely overfitted)")
            print(f"\n{report}")

        return model, y_pred, y_proba, le, class_names, X_clean, y_clean, scaler

In [ ]:
# ── 11.2  PRIMARY: Train binary models (the advisor's approach) ────────────
#    Binary classifier produces sigmoid P(Scale) which we then map
#    to 4 recommendation tiers via thresholds.
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  PHASE 2: XGBoost MODEL TRAINING")
print("  PRIMARY: Binary sigmoid -> 4-class mapping (advisor, March 13)")
print("=" * 70)

# CLIENT_A — Binary (PRIMARY)
# v8.1: Use CLIENT_A DAILY if available (986 rows vs 84 monthly)
# Falls back to monthly if daily is not ready
model_client_a_bin = None
proba_client_a_bin = None
X_client_a_clean = None
y_client_a_bin_clean = None
scaler_client_a = None
client_a_primary_source = None

if (CLIENT_A_DAILY_MODEL_READY and len(X_client_a_daily) > 50
        and y_client_a_daily_bin is not None and y_client_a_daily_bin.notna().sum() > 50):
    # DAILY is the primary source for CLIENT_A
    client_a_primary_source = 'daily'
    print(f"\n--- CLIENT_A: Binary Classification [PRIMARY — DAILY DATA] ---")
    print(f"    Using {len(X_client_a_daily)} daily rows (vs {len(X_client_a)} monthly rows)")
    print(f"    Benchmark: ${benchmarks['CLIENT_A_DAILY_CPL_weighted']:.2f} (daily-specific)")
    (model_client_a_bin, pred_client_a_bin, proba_client_a_bin, le_client_a_bin,
     classes_client_a_bin, X_client_a_clean, y_client_a_bin_clean, scaler_client_a) = train_xgboost_kfold(
        X_client_a_daily, y_client_a_daily_bin,
        model_name='CLIENT_A Binary XGBoost [PRIMARY — DAILY]', is_binary=True
    )

    # Also train the monthly model as comparison
    print(f"\n--- CLIENT_A: Binary Classification [COMPARISON — MONTHLY DATA] ---")
    if y_client_a_bin is not None:
        (model_client_a_bin_monthly, pred_client_a_bin_monthly, proba_client_a_bin_monthly,
         le_client_a_bin_monthly, classes_client_a_bin_monthly,
         X_client_a_clean_monthly, y_client_a_bin_clean_monthly,
         scaler_client_a_monthly) = train_xgboost_kfold(
            X_client_a, y_client_a_bin,
            model_name='CLIENT_A Binary XGBoost [COMPARISON — MONTHLY]', is_binary=True
        )

elif y_client_a_bin is not None:
    # Fallback to monthly
    client_a_primary_source = 'monthly'
    print(f"\n--- CLIENT_A: Binary Classification [PRIMARY — MONTHLY FALLBACK] ---")
    print(f"    CLIENT_A daily not available. Using {len(X_client_a)} monthly rows.")
    (model_client_a_bin, pred_client_a_bin, proba_client_a_bin, le_client_a_bin,
     classes_client_a_bin, X_client_a_clean, y_client_a_bin_clean, scaler_client_a) = train_xgboost_kfold(
        X_client_a, y_client_a_bin, model_name='CLIENT_A Binary XGBoost [PRIMARY — MONTHLY]', is_binary=True
    )

# CLIENT_C — Binary (PRIMARY)
model_client_c_bin = None
proba_client_c_bin = None
X_client_c_clean = None
y_client_c_bin_clean = None
scaler_client_c = None
pred_client_c_bin = None
le_client_c_bin = None
classes_client_c_bin = None

if CLIENT_C_MODEL_READY and len(X_client_c) > 0 and y_client_c_bin is not None and y_client_c_bin.notna().sum() > 50:
    print(f"\n--- CLIENT_C: Binary Classification [PRIMARY] ---")
    print(f"    Using {len(X_client_c)} daily rows")
    print(f"    Benchmark: ${benchmarks['CLIENT_C_CPT_weighted']:.2f} (CPT)")
    (model_client_c_bin, pred_client_c_bin, proba_client_c_bin, le_client_c_bin,
     classes_client_c_bin, X_client_c_clean, y_client_c_bin_clean, scaler_client_c) = train_xgboost_kfold(
        X_client_c, y_client_c_bin,
        model_name='CLIENT_C Binary XGBoost [PRIMARY]', is_binary=True
    )
else:
    print("\n--- CLIENT_C: Binary Classification SKIPPED (no purchase data) ---")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLIENT_B FULL PIPELINE — Benchmark, Label, Feature Engineer
# Paste this BEFORE the CLIENT_B binary model training cell.
# This uses variables already in memory: cleaned, benchmarks,
# assign_recommendation, assign_binary_label, add_optimise_signals,
# engineer_features, CRITICAL_FEATURE_COLS, WB_PALETTE
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  CLIENT_B — FULL PIPELINE (v10)")
print("=" * 70)

# ── Step 1: Benchmark ────────────────────────────────────────────────────
client_b_data = cleaned['CLIENT_B']
client_b_has_purchases = 'purchases' in client_b_data.columns and client_b_data['purchases'].notna().sum() > 0
CLIENT_B_MODEL_READY = False

if client_b_has_purchases:
    client_b_valid = client_b_data[client_b_data['purchases'] > 0].copy()
    client_b_weighted_cpt = client_b_valid['spend'].sum() / client_b_valid['purchases'].sum()
    benchmarks['CLIENT_B_CPT_weighted'] = client_b_weighted_cpt
    client_b_median_roas = client_b_valid['roas'].median() if 'roas' in client_b_valid.columns and client_b_valid['roas'].notna().any() else None
    if client_b_median_roas is not None:
        benchmarks['CLIENT_B_ROAS_median'] = client_b_median_roas
    print(f"CLIENT_B Benchmark — Weighted avg CPT: ${client_b_weighted_cpt:.2f}  ({len(client_b_valid)} rows with purchases)")
else:
    client_b_valid = client_b_data.copy()
    print(f"CLIENT_B — No purchase data. Cannot proceed.")

# ── Step 2: Label ────────────────────────────────────────────────────────
client_b_labelled = pd.DataFrame()

if client_b_has_purchases and 'CLIENT_B_CPT_weighted' in benchmarks:
    desired_client_b_cols = [
        'campaign_name', 'adset_name', 'ad_name',
        'date_start', 'year_month', 'day_of_week_num',
        'spend', 'impressions', 'reach', 'frequency', 'cpm',
        'link_clicks', 'ctr_link', 'cpc_link',
        'purchases', 'cost_per_purchase', 'roas'
    ]
    available_client_b_cols = [col for col in desired_client_b_cols if col in client_b_valid.columns]
    client_b_labelled = client_b_valid[available_client_b_cols].copy()

    if 'date_end' in client_b_valid.columns:
        client_b_labelled['date_end'] = client_b_valid['date_end'].values

    B_client_b = benchmarks['CLIENT_B_CPT_weighted']
    client_b_labelled['recommendation'] = client_b_labelled['cost_per_purchase'].apply(
        lambda x: assign_recommendation(x, B_client_b)
    )
    client_b_labelled['recommendation_binary'] = client_b_labelled['cost_per_purchase'].apply(
        lambda x: assign_binary_label(x, B_client_b)
    )
    client_b_labelled['_client'] = 'CLIENT_B'
    client_b_labelled['_granularity'] = 'daily'

    if client_b_labelled['recommendation'].notna().any():
        CLIENT_B_MODEL_READY = True
        print(f"\nCLIENT_B Labels assigned ({len(client_b_labelled)} rows, benchmark CPT = ${B_client_b:.2f}):")
        print(client_b_labelled['recommendation'].value_counts().to_string())
        print(f"\nCLIENT_B Binary:")
        print(client_b_labelled['recommendation_binary'].value_counts().to_string())
    else:
        print("CLIENT_B: Labels all NaN — check cost_per_purchase.")
else:
    print("CLIENT_B: Cannot label — no benchmark or no purchase data.")

# ── Step 3: Optimise sub-signals ─────────────────────────────────────────
if CLIENT_B_MODEL_READY:
    client_b_labelled = add_optimise_signals(client_b_labelled, goal='purchases')
    print(f"\nCLIENT_B optimise sub-signals added.")

# ── Step 4: Feature engineering ──────────────────────────────────────────
X_client_b = pd.DataFrame()
y_client_b = pd.Series(dtype=str)
y_client_b_bin = None
client_b_features = []

if CLIENT_B_MODEL_READY and len(client_b_labelled) > 0:
    print(f"\n=== CLIENT_B Feature Engineering ===")
    client_b_target = 'cost_per_purchase' if 'cost_per_purchase' in client_b_labelled.columns else 'cpm'
    X_client_b, y_client_b, y_client_b_bin, client_b_features, client_b_labelled = engineer_features(
        client_b_labelled, target_col=client_b_target, critical_cols=CRITICAL_FEATURE_COLS
    )
    print(f"\nCLIENT_B ready for model training: {len(X_client_b)} rows x {X_client_b.shape[1]} features")
else:
    print("CLIENT_B: Feature engineering skipped.")

print("\n" + "=" * 70)
print(f"  CLIENT_B_MODEL_READY = {CLIENT_B_MODEL_READY}")
if CLIENT_B_MODEL_READY:
    print(f"  X_client_b: {X_client_b.shape}  |  y_client_b_bin non-null: {y_client_b_bin.notna().sum()}")
print("=" * 70)

In [ ]:
# ── CLIENT_B — Binary (PRIMARY) ────────────────────────────────────────────────
# v10: CLIENT_B integrated. Same approach as CLIENT_C (purchase-based, daily data).
model_client_b_bin = None
proba_client_b_bin = None
X_client_b_clean = None
y_client_b_bin_clean = None
scaler_client_b = None
pred_client_b_bin = None
le_client_b_bin = None
classes_client_b_bin = None

if CLIENT_B_MODEL_READY and len(X_client_b) > 50 and y_client_b_bin is not None and y_client_b_bin.notna().sum() > 50:
    print(f"\n--- CLIENT_B: Binary Classification [PRIMARY] ---")
    print(f"    Using {len(X_client_b)} daily rows")
    print(f"    Benchmark: ${benchmarks['CLIENT_B_CPT_weighted']:.2f} (CPT)")
    (model_client_b_bin, pred_client_b_bin, proba_client_b_bin, le_client_b_bin,
     classes_client_b_bin, X_client_b_clean, y_client_b_bin_clean, scaler_client_b) = train_xgboost_kfold(
        X_client_b, y_client_b_bin,
        model_name='CLIENT_B Binary XGBoost [PRIMARY]', is_binary=True
    )
else:
    print("\n--- CLIENT_B: Binary Classification SKIPPED (insufficient data) ---")

In [ ]:
# ── 11.3  Sigmoid-to-4-class mapping (the advisor's framework) ──────────
#
#    Advisor guidance (March 13): use the binary model's predicted
#    probability. A high probability is a clear scale-up, a middle band
#    means maintain, and lower probabilities fall to the remaining tiers.
#
#    We use the binary model's P(Scale) to derive four tiers:
#      P(Scale) > 0.70  ->  Scale Up
#      P(Scale) 0.40-0.70 -> Maintain
#      P(Scale) 0.20-0.40 -> Optimise
#      P(Scale) < 0.20  ->  Pause
#
#    The P(Scale) value IS the confidence score.
# ──────────────────────────────────────────────────────────────────

# Thresholds (adjustable — to be validated with the client analyst and the client lead)
SIGMOID_THRESHOLDS = {
    'scale_up': 0.70,   # P(Scale) above this -> Scale Up
    'maintain': 0.40,   # P(Scale) between maintain and scale_up -> Maintain
    'optimise': 0.20,   # P(Scale) between optimise and maintain -> Optimise
    # Below optimise threshold -> Pause
}

def sigmoid_to_4class(p_scale, thresholds=SIGMOID_THRESHOLDS):
    """
    Map binary sigmoid probability P(Scale) to 4 recommendation tiers.

    This is the advisor's core suggestion: train binary, use probabilities
    as a continuous signal, and bin into actionable categories.

    Parameters
    ----------
    p_scale : float
        Probability of 'Scale' from the binary classifier (0 to 1).
    thresholds : dict
        Threshold values for each tier boundary.

    Returns
    -------
    recommendation : str
        One of 'Scale Up', 'Maintain', 'Optimise', 'Pause'.
    """
    if pd.isna(p_scale) or pd.isna(thresholds) or pd.isna(thresholds['scale_up']):
        return np.nan
    if p_scale >= thresholds['scale_up']:
        return 'Scale Up'
    elif p_scale >= thresholds['maintain']:
        return 'Maintain'
    elif p_scale >= thresholds['optimise']:
        return 'Optimise'
    else:
        return 'Pause'

print("\n" + "=" * 70)
print("  SIGMOID-TO-4-CLASS MAPPING (the advisor's framework)")
print("=" * 70)
print(f"  Thresholds: Scale Up >= {SIGMOID_THRESHOLDS['scale_up']:.2f}  |  "
      f"Maintain >= {SIGMOID_THRESHOLDS['maintain']:.2f}  |  "
      f"Optimise >= {SIGMOID_THRESHOLDS['optimise']:.2f}  |  "
      f"Pause < {SIGMOID_THRESHOLDS['optimise']:.2f}")

In [ ]:
# ── Apply mapping to CLIENT_B ──────────────────────────────────────────────────
# v10: CLIENT_B sigmoid mapping. Same structure as CLIENT_C.
sigmoid_client_b = None
if model_client_b_bin is not None and proba_client_b_bin is not None:
    scale_idx_client_b = list(classes_client_b_bin).index('Scale') if 'Scale' in classes_client_b_bin else 1
    p_scale_client_b = proba_client_b_bin[:, scale_idx_client_b]
    sigmoid_rec_client_b = pd.Series(p_scale_client_b).apply(sigmoid_to_4class).values

    sigmoid_client_b = pd.DataFrame({
        'p_scale': p_scale_client_b,
        'sigmoid_recommendation': sigmoid_rec_client_b,
        'confidence': p_scale_client_b,
        'binary_prediction': pred_client_b_bin,
    }, index=X_client_b_clean.index)

    for col in ['campaign_name', 'adset_name', 'year_month', 'cost_per_purchase']:
        if col in client_b_labelled.columns:
            try:
                sigmoid_client_b[col] = client_b_labelled.loc[X_client_b_clean.index, col].values
            except KeyError:
                pass

    sigmoid_client_b['rule_based_label'] = y_client_b.loc[X_client_b_clean.index].values
    sigmoid_client_b['confidence_level'] = pd.cut(
        sigmoid_client_b['p_scale'],
        bins=[0, 0.20, 0.40, 0.70, 1.0],
        labels=['Low (Pause zone)', 'Below mid (Optimise zone)',
                'Moderate (Maintain zone)', 'High (Scale Up zone)']
    )

    print(f"\nCLIENT_B — Sigmoid-to-4-Class Distribution:")
    print(sigmoid_client_b['sigmoid_recommendation'].value_counts().to_string())
    print(f"\nCLIENT_B — P(Scale) Statistics:")
    print(f"  Mean:   {p_scale_client_b.mean():.3f}")
    print(f"  Median: {np.median(p_scale_client_b):.3f}")

if sigmoid_client_b is not None:
    sigmoid_client_b.to_csv(OUTPUT_DIR / 'client_b_sigmoid_recommendations.csv', index=False)
    print(f"CLIENT_B sigmoid recommendations exported")

In [ ]:
# ── Apply mapping to CLIENT_A ──────────
sigmoid_client_a = None
if client_a_primary_source:
        print(f"\n  CLIENT_A sigmoid mapping source: {client_a_primary_source.upper()} data")
if model_client_a_bin is not None and proba_client_a_bin is not None:
    # Extract P(Scale) ─ the probability of the positive class
    # In binary:logistic, predict_proba returns [P(class0), P(class1)]
    # We need P(Scale), which is the class labelled 'Scale' by LabelEncoder
    scale_idx_client_a = list(classes_client_a_bin).index('Scale') if 'Scale' in classes_client_a_bin else 1
    p_scale_client_a = proba_client_a_bin[:, scale_idx_client_a]

    # Map to 4-class recommendations
    sigmoid_rec_client_a = pd.Series(p_scale_client_a).apply(sigmoid_to_4class).values

    sigmoid_client_a = pd.DataFrame({
        'p_scale': p_scale_client_a,
        'sigmoid_recommendation': sigmoid_rec_client_a,
        'confidence': p_scale_client_a,  # P(Scale) IS the confidence
        'binary_prediction': pred_client_a_bin,
    }, index=X_client_a_clean.index)

    # Add campaign context
    # Determine the correct source dataframe based on client_a_primary_source
    if client_a_primary_source == 'daily':
        source_df_for_context = client_a_daily_labelled
    else:
        source_df_for_context = client_a_labelled

    for col in ['campaign_name', 'adset_name', 'year_month', 'cpl_calc']:
        if col in source_df_for_context.columns:
            try:
                sigmoid_client_a[col] = source_df_for_context.loc[X_client_a_clean.index, col].values
            except KeyError:
                pass # This indicates an index mismatch, should ideally not happen if logic is correct

    # Rule-based label from the matching source
    if client_a_primary_source == 'daily':
        # Use y_client_a_daily for daily data, which corresponds to client_a_daily_labelled and X_client_a_clean
        sigmoid_client_a['rule_based_label'] = y_client_a_daily.loc[X_client_a_clean.index].values
    else:
        # Use y_client_a for monthly data, which corresponds to client_a_labelled
        sigmoid_client_a['rule_based_label'] = y_client_a.loc[X_client_a_clean.index].values

    # Confidence categories
    sigmoid_client_a['confidence_level'] = pd.cut(
        sigmoid_client_a['p_scale'],
        bins=[0, 0.20, 0.40, 0.70, 1.0],
        labels=['Low (Pause zone)', 'Below mid (Optimise zone)',
                'Moderate (Maintain zone)', 'High (Scale Up zone)']
    )

    print(f"\nCLIENT_A ─ Sigmoid-to-4-Class Distribution:")
    print(sigmoid_client_a['sigmoid_recommendation'].value_counts().to_string())
    print(f"\nCLIENT_A ─ P(Scale) Statistics:")
    print(f"  Mean:   {p_scale_client_a.mean():.3f}")
    print(f"  Median: {np.median(p_scale_client_a):.3f}")
    print(f"  Min:    {p_scale_client_a.min():.3f}")
    print(f"  Max:    {p_scale_client_a.max():.3f}")

    # Visualise
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: P(Scale) distribution with threshold lines
    axes[0].hist(p_scale_client_a, bins=20, color=WB_PALETTE['CLIENT_A'],
                 edgecolor='white', alpha=0.8)
    # Y-axis limited to 150 to show distribution detail
    # Scale Up spike (560 rows at P≈1.0) annotated separately
    axes[0].set_ylim(0, 150)
    axes[0].annotate(f'Scale Up spike: 560 rows at P≈1.0',
                xy=(0.72, 140), fontsize=8, color=WB_PALETTE['Scale Up'])

    for label, thresh in SIGMOID_THRESHOLDS.items():
        axes[0].axvline(thresh, linestyle='--', linewidth=1.2,
                       color=WB_PALETTE['ref_line'], alpha=0.7)
        axes[0].text(thresh + 0.01, axes[0].get_ylim()[1] * 0.9,
                    f'{label}\n({thresh})', fontsize=8, va='top')
    axes[0].set_title('CLIENT_A ─ P(Scale) Distribution\n[Binary Sigmoid Output]',
                     fontweight='bold')
    axes[0].set_xlabel('P(Scale)')
    axes[0].set_ylabel('Count')

    # Panel 2: Derived 4-class distribution
    rec_counts = sigmoid_client_a['sigmoid_recommendation'].value_counts()
    rec_colors = [WB_PALETTE.get(c, '#999') for c in rec_counts.index]
    bars = axes[1].bar(rec_counts.index, rec_counts.values,
                      color=rec_colors, edgecolor='white')
    axes[1].bar_label(bars)
    axes[1].set_title('CLIENT_A ─ Derived 4-Class Recommendations\n[From Binary Sigmoid]',
                     fontweight='bold')
    axes[1].set_ylabel('Count')

    # Panel 3: Comparison ─ sigmoid-derived vs rule-based labels
    comparison = pd.crosstab(sigmoid_client_a['rule_based_label'],
                            sigmoid_client_a['sigmoid_recommendation'])
    comparison.plot(kind='bar', ax=axes[2], color=[WB_PALETTE.get(c, '#999')
                   for c in comparison.columns], edgecolor='white')
    axes[2].set_title('CLIENT_A ─ Rule-Based vs Sigmoid-Derived Labels',
                     fontweight='bold')
    axes[2].set_xlabel('Rule-Based Label')
    axes[2].set_ylabel('Count')
    axes[2].legend(title='Sigmoid-Derived', fontsize=8)
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

# ── Apply mapping to CLIENT_C ──────────────────
sigmoid_client_c = None
if model_client_c_bin is not None and proba_client_c_bin is not None:
    scale_idx_client_c = list(classes_client_c_bin).index('Scale') if 'Scale' in classes_client_c_bin else 1
    p_scale_client_c = proba_client_c_bin[:, scale_idx_client_c]

    sigmoid_rec_client_c = pd.Series(p_scale_client_c).apply(sigmoid_to_4class).values

    sigmoid_client_c = pd.DataFrame({
        'p_scale': p_scale_client_c,
        'sigmoid_recommendation': sigmoid_rec_client_c,
        'confidence': p_scale_client_c,
        'binary_prediction': pred_client_c_bin,
    }, index=X_client_c_clean.index)

    # Add campaign context for CLIENT_C
    for col in ['campaign_name', 'adset_name', 'year_month', 'cost_per_purchase']:
        if col in client_c_labelled.columns:
            try:
                sigmoid_client_c[col] = client_c_labelled.loc[X_client_c_clean.index, col].values
            except KeyError:
                pass  # index mismatch ─ skip this context column

    sigmoid_client_c['rule_based_label'] = y_client_c.loc[X_client_c_clean.index].values

    sigmoid_client_c['confidence_level'] = pd.cut(
        sigmoid_client_c['p_scale'],
        bins=[0, 0.20, 0.40, 0.70, 1.0],
        labels=['Low (Pause zone)', 'Below mid (Optimise zone)',
                'Moderate (Maintain zone)', 'High (Scale Up zone)']
    )

    print(f"\nCLIENT_C ─ Sigmoid-to-4-Class Distribution:")
    print(sigmoid_client_c['sigmoid_recommendation'].value_counts().to_string())
    print(f"\nCLIENT_C ─ P(Scale) Statistics:")
    print(f"  Mean:   {p_scale_client_c.mean():.3f}")
    print(f"  Median: {np.median(p_scale_client_c):.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(p_scale_client_c, bins=25, color=WB_PALETTE['CLIENT_C'],
                 edgecolor='white', alpha=0.8)
    # Y-axis limited to 200 to show distribution detail
    # Scale Up spike (2236 rows at P≈1.0) annotated separately
    axes[0].set_ylim(0, 200)
    axes[0].annotate(f'Scale Up spike: 2236 rows at P≈1.0',
                xy=(0.72, 185), fontsize=8, color=WB_PALETTE['CLIENT_C'])

    for label, thresh in SIGMOID_THRESHOLDS.items():
        axes[0].axvline(thresh, linestyle='--', linewidth=1.2,
                       color=WB_PALETTE['ref_line'], alpha=0.7)
    axes[0].set_title('CLIENT_C ─ P(Scale) Distribution\n[Binary Sigmoid Output]',
                     fontweight='bold')
    axes[0].set_xlabel('P(Scale)')

    rec_counts_e = sigmoid_client_c['sigmoid_recommendation'].value_counts()
    rec_colors_e = [WB_PALETTE.get(c, '#999') for c in rec_counts_e.index]
    bars_e = axes[1].bar(rec_counts_e.index, rec_counts_e.values,
                        color=rec_colors_e, edgecolor='white')
    axes[1].bar_label(bars_e)
    axes[1].set_title('CLIENT_C ─ Derived 4-Class Recommendations\n[From Binary Sigmoid]',
                     fontweight='bold')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# Export sigmoid-derived recommendations (PRIMARY output)
if sigmoid_client_a is not None:
    sigmoid_client_a.to_csv(OUTPUT_DIR / 'client_a_sigmoid_recommendations.csv', index=False)
    print(f"\nCLIENT_A sigmoid recommendations exported")
if sigmoid_client_c is not None:
    sigmoid_client_c.to_csv(OUTPUT_DIR / 'client_c_sigmoid_recommendations.csv', index=False)
    print(f"CLIENT_C sigmoid recommendations exported")

In [ ]:
# ── 11.4  SECONDARY: Train 4-class models (comparison) ───────────
#    Retained to compare against the sigmoid-derived labels.
#    NOT the primary deliverable per mentor guidance.
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  SECONDARY: 4-Class XGBoost (comparison only)")
print("=" * 70)

# CLIENT_A — 4-class (comparison) — use daily if available for fair comparison
# with the binary model (both trained on same data)
if CLIENT_A_DAILY_MODEL_READY and len(X_client_a_daily) > 50:
    print("\n--- CLIENT_A: 4-Class [COMPARISON — DAILY] (CPL thresholds) ---")
    (model_client_a_4c, pred_client_a_4c, proba_client_a_4c, le_client_a_4c,
     classes_client_a_4c, X_client_a_clean_4c, y_client_a_clean, scaler_client_a_4c) = train_xgboost_kfold(
        X_client_a_daily, y_client_a_daily,
        model_name='CLIENT_A 4-Class XGBoost [COMPARISON — DAILY]',
        is_binary=False
    )
else:
    print("\n--- CLIENT_A: 4-Class [COMPARISON — MONTHLY] (CPL thresholds) ---")
    (model_client_a_4c, pred_client_a_4c, proba_client_a_4c, le_client_a_4c,
     classes_client_a_4c, X_client_a_clean_4c, y_client_a_clean, scaler_client_a_4c) = train_xgboost_kfold(
        X_client_a, y_client_a,
        model_name='CLIENT_A 4-Class XGBoost [COMPARISON — MONTHLY]',
        is_binary=False
    )

# CLIENT_C — 4-class (comparison)
model_client_c_4c = None
proba_client_c_4c = None
if CLIENT_C_MODEL_READY and len(X_client_c) > 0 and y_client_c.notna().sum() > 5:
    print("\n--- CLIENT_C: 4-Class [COMPARISON] (CPT thresholds) ---")
    (model_client_c_4c, pred_client_c_4c, proba_client_c_4c, le_client_c_4c,
     classes_client_c_4c, X_client_c_clean_4c, y_client_c_clean, scaler_client_c_4c) = train_xgboost_kfold(
        X_client_c, y_client_c, model_name='CLIENT_C 4-Class XGBoost [COMPARISON]', is_binary=False
    )
else:
    print("\n--- CLIENT_C: 4-Class SKIPPED (no purchase data) ---")


In [ ]:
# ── 11.5  Confusion matrices ─────────────────────────────────────────
    # Binary confusion matrices (PRIMARY)
print("\n--- PRIMARY: Binary Confusion Matrices ---")
n_bin_plots = 1 + (1 if model_client_c_bin is not None else 0)
fig, axes = plt.subplots(1, n_bin_plots, figsize=(7 * n_bin_plots, 5))
if n_bin_plots == 1:
    axes = [axes]

if model_client_a_bin is not None:
    cm_client_a_bin = confusion_matrix(y_client_a_bin_clean, pred_client_a_bin, labels=classes_client_a_bin)
    sns.heatmap(cm_client_a_bin, annot=True, fmt='d', cmap=WB_CMAP,
                xticklabels=classes_client_a_bin, yticklabels=classes_client_a_bin, ax=axes[0])
    axes[0].set_title('CLIENT_A — Binary Confusion Matrix [PRIMARY]', fontweight='bold')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

if model_client_c_bin is not None and n_bin_plots > 1:
    cm_client_c_bin = confusion_matrix(y_client_c_bin_clean, pred_client_c_bin, labels=classes_client_c_bin)
    sns.heatmap(cm_client_c_bin, annot=True, fmt='d', cmap=WB_CMAP,
                xticklabels=classes_client_c_bin, yticklabels=classes_client_c_bin, ax=axes[1])
    axes[1].set_title('CLIENT_C — Binary Confusion Matrix [PRIMARY]', fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# 4-class confusion matrices (COMPARISON)
print("\n--- SECONDARY: 4-Class Confusion Matrices (comparison) ---")
if model_client_c_4c is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
else:
    fig, axes = plt.subplots(1, 1, figsize=(7, 5))
    axes = [axes]

cm_client_a = confusion_matrix(y_client_a_clean, pred_client_a_4c, labels=classes_client_a_4c)
sns.heatmap(cm_client_a, annot=True, fmt='d', cmap=WB_CMAP,
            xticklabels=classes_client_a_4c, yticklabels=classes_client_a_4c, ax=axes[0])
axes[0].set_title('CLIENT_A — 4-Class Confusion Matrix [COMPARISON]', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

if model_client_c_4c is not None and len(axes) > 1:
    cm_client_c = confusion_matrix(y_client_c_clean, pred_client_c_4c, labels=classes_client_c_4c)
    sns.heatmap(cm_client_c, annot=True, fmt='d', cmap=WB_CMAP,
                xticklabels=classes_client_c_4c, yticklabels=classes_client_c_4c, ax=axes[1])
    axes[1].set_title('CLIENT_C — 4-Class Confusion Matrix [COMPARISON]', fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 12. Feature Importance & Model Evaluation

  Feature importance is a key deliverable for the agency partner. It shows which
  advertising metrics most influence the recommendation outcome, this may
  differ from the team's intuition and provides real analytical value.

  v8: Feature importance extracted from BINARY models (primary) and
  4-class models (comparison). the advisor noted feature importance is probably
  the most useful output for the agency partner even beyond the classification itself.

In [ ]:
# ── 12.1  Feature importance extraction ──────────────────────────────
def plot_feature_importance(model, feature_names, title, color, top_n=15):
    """Plot XGBoost feature importance (gain-based) as a horizontal bar chart."""
    importances = model.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.35)))
    ax.barh(feat_imp['feature'], feat_imp['importance'], color=color,
            edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Feature Importance (Gain)')

    for i, (val, name) in enumerate(zip(feat_imp['importance'], feat_imp['feature'])):
        ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()
    return feat_imp


In [ ]:
print("\n" + "=" * 70)
print("  FEATURE IMPORTANCE — KEY DELIVERABLE FOR the agency partner")
print("=" * 70)

# PRIMARY: Feature importance from binary models
print("\n--- CLIENT_A (Binary Model — PRIMARY) — Top Features ---")
imp_client_a = None
if model_client_a_bin is not None:
    imp_client_a = plot_feature_importance(
        model_client_a_bin,
        feature_names=X_client_a_clean.columns.tolist(),
        title='CLIENT_A — Feature Importance (Binary Scale/Not Scale) [PRIMARY]',
        color=WB_PALETTE['CLIENT_A']
    )
    print(imp_client_a.sort_values('importance', ascending=False).to_string(index=False))

imp_client_c = None
if model_client_c_bin is not None:
    print("\n--- CLIENT_C (Binary Model — PRIMARY) — Top Features ---")
    imp_client_c = plot_feature_importance(
        model_client_c_bin,
        feature_names=X_client_c_clean.columns.tolist(),
        title='CLIENT_C — Feature Importance (Binary Scale/Not Scale) [PRIMARY]',
        color=WB_PALETTE['CLIENT_C']
    )
    print(imp_client_c.sort_values('importance', ascending=False).to_string(index=False))


In [ ]:
# SECONDARY: Feature importance from 4-class models (comparison)
print("\n--- CLIENT_A (4-Class Model — COMPARISON) — Top Features ---")
imp_client_a_4c = plot_feature_importance(
    model_client_a_4c,
    feature_names=X_client_a_clean_4c.columns.tolist(),
    title='CLIENT_A — Feature Importance (4-Class) [COMPARISON]',
    color=WB_PALETTE['CLIENT_A']
)

imp_client_c_4c = None
if model_client_c_4c is not None:
    print("\n--- CLIENT_C (4-Class Model — COMPARISON) — Top Features ---")
    imp_client_c_4c = plot_feature_importance(
        model_client_c_4c,
        feature_names=X_client_c_clean_4c.columns.tolist(),
        title='CLIENT_C — Feature Importance (4-Class) [COMPARISON]',
        color=WB_PALETTE['CLIENT_C']
    )
else:
    print("\n--- CLIENT_C: Feature importance not available (no purchase data) ---")

In [ ]:
# ── 12.2  Cross-model comparison ─────────────────────────────────────
if imp_client_c is not None:
    print("\n--- Feature Importance Comparison (CLIENT_A vs CLIENT_C) ---")
    print("This shows whether the same metrics drive performance across")
    print("different campaign goals (leads vs purchases).\n")

    imp_client_a_top = imp_client_a.sort_values('importance', ascending=False).head(10).copy()
    imp_client_a_top.columns = ['feature', 'CLIENT_A_importance']
    imp_client_c_top = imp_client_c.sort_values('importance', ascending=False).head(10).copy()
    imp_client_c_top.columns = ['feature', 'CLIENT_C_importance']

    comparison = imp_client_a_top.merge(imp_client_c_top, on='feature', how='outer').fillna(0)
    comparison = comparison.sort_values('CLIENT_A_importance', ascending=False)
    print(comparison.to_string(index=False))

In [ ]:
# ── 12.3  Export model artefacts ─────────────────────────────────────
if imp_client_a is not None:
    imp_client_a.to_csv(OUTPUT_DIR / 'feature_importance_client_a_binary.csv', index=False)
if imp_client_c is not None:
    imp_client_c.to_csv(OUTPUT_DIR / 'feature_importance_client_c_binary.csv', index=False)
imp_client_a_4c.to_csv(OUTPUT_DIR / 'feature_importance_client_a_4class.csv', index=False)
if imp_client_c_4c is not None:
    imp_client_c_4c.to_csv(OUTPUT_DIR / 'feature_importance_client_c_4class.csv', index=False)

eval_summary = {
    'primary_approach': 'Binary XGBoost (sigmoid) -> 4-class via probability thresholds (advisor framework)',
    'secondary_approach': '4-class XGBoost (softprob) — comparison only',
    'client_a_binary_primary': {
        'accuracy': accuracy_score(y_client_a_bin_clean, pred_client_a_bin) if model_client_a_bin is not None else 'NOT TRAINED',
        'f1_macro': f1_score(
            le_client_a_bin.transform(y_client_a_bin_clean),
            le_client_a_bin.transform(pred_client_a_bin),
            average='macro'
        ) if model_client_a_bin is not None else 'NOT TRAINED',
        'n_samples': len(y_client_a_bin_clean) if y_client_a_bin_clean is not None else 0,
        'n_features': X_client_a_clean.shape[1] if X_client_a_clean is not None else 0,
        'sigmoid_thresholds': SIGMOID_THRESHOLDS,
    },
    'client_a_4class_comparison': {
        'accuracy': accuracy_score(y_client_a_clean, pred_client_a_4c),
        'f1_macro': f1_score(
            le_client_a_4c.transform(y_client_a_clean),
            le_client_a_4c.transform(pred_client_a_4c),
            average='macro'
        ),
        'n_samples': len(y_client_a_clean),
        'n_features': X_client_a_clean_4c.shape[1],
    },
    'threshold_status': 'Defaults used. Sigmoid thresholds (0.70/0.40/0.20) and '
                        'rule-based thresholds (70/110/160%) NOT yet validated with the client analyst/the client lead.',
    'data_decision': 'Daily data received March 17. All clients on daily data. '
                 'CLIENT_A daily (986 rows with leads) replaced monthly (84 rows) as primary. '
                 'Monthly retained as comparison.',
    'targets': {'CLIENT_A': 'CPL (Cost Per Lead)', 'CLIENT_C': 'CPT (Cost Per Purchase)'},
    'imputation_method': 'Median fill for missing numeric features. NOTE: KNN imputation was '
                         'discussed with the advisor but median is currently used. Imputed rows are '
                         'not individually flagged — this should be documented when presenting.',
}
if model_client_c_bin is not None:
    eval_summary['client_c_binary_primary'] = {
        'accuracy': accuracy_score(y_client_c_bin_clean, pred_client_c_bin),
        'f1_macro': f1_score(
            le_client_c_bin.transform(y_client_c_bin_clean),
            le_client_c_bin.transform(pred_client_c_bin),
            average='macro'
        ),
        'n_samples': len(y_client_c_bin_clean),
        'n_features': X_client_c_clean.shape[1],
    }
if model_client_c_4c is not None:
    eval_summary['client_c_4class_comparison'] = {
        'accuracy': accuracy_score(y_client_c_clean, pred_client_c_4c),
        'f1_macro': f1_score(
            le_client_c_4c.transform(y_client_c_clean),
            le_client_c_4c.transform(pred_client_c_4c),
            average='macro'
        ),
        'n_samples': len(y_client_c_clean),
        'n_features': X_client_c_clean_4c.shape[1],
    }
else:
    eval_summary['client_c_4class_comparison'] = 'NOT TRAINED — purchase columns missing'
with open(OUTPUT_DIR / 'model_evaluation.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print("\nModel artefacts exported to:", OUTPUT_DIR)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  PHASE 3 — CONFIDENCE VALIDATION, ALERTING RECOMMENDATIONS & REPORT
# ═══════════════════════════════════════════════════════════════════════════

## 13. Phase 3 - Confidence Validation (Binary Sigmoid Framework)

### Binary sigmoid is the probability framework

The primary confidence score is P(Scale) from the binary sigmoid,
which was already computed and mapped to 4 tiers in Section 11.3.

This section validates whether the binary sigmoid confidence is actually
informative — i.e., does higher P(Scale) correlate with better outcomes?

"The team can trust the model's recommendation when
the confidence score is high, but recommendations with low confidence
require manual review."

In [ ]:
# ── 13.1  Confidence validation — CLIENT_A (binary sigmoid) ────────────────────

print("\n" + "=" * 70)
print("  CONFIDENCE VALIDATION — BINARY SIGMOID (PRIMARY)")
print("=" * 70)

if sigmoid_client_a is not None:
    # Compare sigmoid-derived labels against rule-based labels
    sigmoid_client_a['matches_rule'] = (
        sigmoid_client_a['sigmoid_recommendation'] == sigmoid_client_a['rule_based_label']
    )

    print("\nCLIENT_A — Sigmoid vs Rule-Based Label Agreement:")
    agreement_rate = sigmoid_client_a['matches_rule'].mean()
    print(f"  Overall agreement: {agreement_rate:.1%}")

    print("\nCLIENT_A — Agreement by confidence zone:")
    conf_agreement = sigmoid_client_a.groupby('confidence_level')['matches_rule'].agg(['mean', 'count'])
    conf_agreement.columns = ['agreement_rate', 'n_rows']
    print(conf_agreement.to_string())

    print(f"\nCLIENT_A — P(Scale) by sigmoid-derived recommendation:")
    print(sigmoid_client_a.groupby('sigmoid_recommendation')['p_scale'].describe()[
        ['count', 'mean', 'min', 'max']
    ].round(3).to_string())

    # NOTE: CLIENT_A daily model — confidence validation confirms sigmoid is informative
    print(f"\n  NOTE: CLIENT_A has 986 daily rows with leads.")
    print(f"  Confidence validation shows high-confidence predictions (P > 0.70)")
    print(f"  agree with rule-based labels 72.9% of the time vs 37.5% in the")
    print(f"  Optimise zone. Low-confidence cases require manual review by the client analyst.")

    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for rec in ['Scale Up', 'Maintain', 'Optimise', 'Pause']:
        subset = sigmoid_client_a[sigmoid_client_a['sigmoid_recommendation'] == rec]['p_scale']
        if len(subset) > 0:
            axes[0].hist(subset, bins=15, alpha=0.6,
                        color=WB_PALETTE.get(rec, '#999'), label=rec, edgecolor='white')
    axes[0].set_title('CLIENT_A — P(Scale) by Derived Recommendation', fontweight='bold')
    axes[0].set_xlabel('P(Scale)')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    # Y-axis limited to 150 to show distribution detail
    # Scale Up spike (560 rows at P≈1.0) annotated separately
    axes[0].set_ylim(0, 150)
    axes[0].annotate('Scale Up spike: 560 rows at P≈1.0',
                xy=(0.72, 140), fontsize=8, color=WB_PALETTE['Scale Up'])

    if len(conf_agreement) > 0:
        zone_colors = [WB_PALETTE['Pause'], WB_PALETTE['Optimise'],
                       WB_PALETTE['Maintain'], WB_PALETTE['Scale Up']]
        bars = axes[1].bar(range(len(conf_agreement)), conf_agreement['agreement_rate'],
                          color=zone_colors[:len(conf_agreement)], edgecolor='white')
        axes[1].set_xticks(range(len(conf_agreement)))
        axes[1].set_xticklabels(conf_agreement.index, rotation=30, ha='right', fontsize=9)
        axes[1].bar_label(bars, fmt=lambda x: f'{x:.1%}')
        axes[1].set_title('CLIENT_A — Agreement with Rule-Based Labels\nby Confidence Zone',
                         fontweight='bold')
        axes[1].set_ylabel('Agreement Rate')
        axes[1].set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 13.2  Confidence validation — CLIENT_C (binary sigmoid) ──────────────────

if sigmoid_client_c is not None:
    print("\n" + "=" * 70)
    print("  CONFIDENCE VALIDATION — CLIENT_C (BINARY SIGMOID)")
    print("=" * 70)

    sigmoid_client_c['matches_rule'] = (
        sigmoid_client_c['sigmoid_recommendation'] == sigmoid_client_c['rule_based_label']
    )

    agreement_rate_client_c = sigmoid_client_c['matches_rule'].mean()
    print(f"\nCLIENT_C — Overall agreement: {agreement_rate_client_c:.1%}")

    print("\nCLIENT_C — Agreement by confidence zone:")
    conf_agreement_client_c = sigmoid_client_c.groupby('confidence_level')['matches_rule'].agg(['mean', 'count'])
    conf_agreement_client_c.columns = ['agreement_rate', 'n_rows']
    print(conf_agreement_client_c.to_string())

    print(f"\n  CLIENT_C has more data ({len(sigmoid_client_c)} rows).")
    print(f"  Week 9 validation confirmed: high confidence = high accuracy for CLIENT_C.")

    fig, ax = plt.subplots(figsize=(7, 5))
    for rec in ['Scale Up', 'Maintain', 'Optimise', 'Pause']:
        subset = sigmoid_client_c[sigmoid_client_c['sigmoid_recommendation'] == rec]['p_scale']
        if len(subset) > 0:
            ax.hist(subset, bins=20, alpha=0.6,
                   color=WB_PALETTE.get(rec, '#999'), label=rec, edgecolor='white')
    ax.set_title('CLIENT_C — P(Scale) by Derived Recommendation', fontweight='bold')
    ax.set_xlabel('P(Scale)')
    ax.set_ylabel('Count')
    ax.legend()

    # Y-axis limited to 200 to show distribution detail
    # Scale Up spike (2236 rows at P≈1.0) annotated separately
    ax.set_ylim(0, 200)
    ax.annotate('Scale Up spike: 2236 rows at P≈1.0',
            xy=(0.72, 185), fontsize=8, color=WB_PALETTE['Scale Up'])

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 13.3  Export confidence data ──────────────────────────────────────────
if sigmoid_client_a is not None:
    sigmoid_client_a.to_csv(OUTPUT_DIR / 'client_a_sigmoid_confidence.csv', index=False)
if sigmoid_client_c is not None:
    sigmoid_client_c.to_csv(OUTPUT_DIR / 'client_c_sigmoid_confidence.csv', index=False)

print("\nConfidence data exported")

Confidence validation summary:

For Client C (larger dataset), higher P(Scale) values reliably predict better
campaign outcomes, confirming that the binary sigmoid framework produces
meaningful confidence scores.

For Client A (smaller dataset), the model may be overconfident. All Client A
recommendations should be treated as indicative rather than definitive,
and low-confidence cases should always be reviewed by the client analyst.

This supports the Week 9 key decision: trust high-confidence
recommendations, manually review low-confidence ones.

## 14. Alerting System — Recommendations for the agency partner

### Week 9 Key Decision:
> "The team will provide recommendations for building an alerting system
> rather than building the system itself."

This section provides documentation and guidance for building a proactive
alerting system, based on the client lead's requirements and market best practices.
The team does not build the system itself due to time constraints and the
absence of live API access.

### the client lead's requirement (March 5 meeting):
> "Some form of proactive notification coming through to the client analyst... an alert
> which says something like: alert, budget alert, Client B,
> this ad set. And then: we recommend pausing or turning off this ad set
> for reason."

### Recommended alerting architecture:

**1. Data source:**
   Connect to Meta Ads API for real-time campaign data. The current POC
   operates on historical exports; live alerting requires API access that
   was not provided to the team.

**2. Monitoring logic:**
   For each campaign/ad set, compare current CPL or CPT against the
   benchmark threshold. Use the binary sigmoid P(Scale) value to classify
   the campaign into one of the 4 recommendation tiers.

**3. Alert severity mapping:**
   - CRITICAL (P(Scale) < 0.20 / Pause zone): Immediate action required.
     Recommend pausing the campaign with specific rationale.
   - WARNING (P(Scale) 0.20-0.40 / Optimise zone): Review and adjust.
     Include sub-signal detail (budget pacing, CPM, frequency fatigue).
   - INFO (P(Scale) > 0.70 / Scale Up zone): Opportunity notification.
     Recommend increasing budget allocation.
   - No alert for Maintain zone (P(Scale) 0.40-0.70) unless confidence
     is unusually low (borderline predictions flagged for review).

**4. Delivery channels (recommended):**
   - Email alerts via SMTP or SendGrid (for the client analyst's daily workflow)
   - Slack notifications via webhook (for real-time team visibility)
   - Dashboard popup in Streamlit or Retool (for visual monitoring)
   - Google Sheets export (simplest option for the client analyst's current workflow)

**5. Alert content (per the client lead's specification):**
   Each alert should include:
   - Severity level (CRITICAL / WARNING / INFO)
   - Affected client, campaign, and ad set
   - Recommended action (Scale Up / Maintain / Optimise / Pause)
   - Rationale (which metric triggered the alert and by how much)
   - Model confidence score (P(Scale) value)
   - Optimise sub-signals where applicable (budget pacing, placement
     cost, creative refresh)

**6. Implementation path:**
   a. Obtain Meta Ads API read access for the agency partner's client accounts
   b. Schedule a daily data pull (cron job or cloud function)
   c. Run the binary XGBoost model on fresh data
   d. Apply sigmoid-to-4-class mapping
   e. Generate alerts for campaigns in Pause or Optimise zones
   f. Deliver via chosen channel

**7. Constraints:**
   - The model was trained on historical data with team-defined thresholds.
     Thresholds must be validated with the client analyst and the client lead before any
     operational alerting.
   - Client A model may be overconfident due to small training set. Client A alerts
     should always include a manual review flag.
   - The team did not have Meta Ads API access. All alerting logic is
     designed as a framework that can connect to live data once API
     access is established.

In [ ]:
# ── 14.1  Alerting system documentation export ────────────────────────────

print("\n" + "=" * 70)
print("  ALERTING SYSTEM — RECOMMENDATIONS FOR the agency partner")
print("=" * 70)

alerting_doc = {
    'section': 'Alerting System Recommendations',
    'status': 'Documentation only (per Week 9 decision — not built as working system)',
    'client_lead_requirement': (
        'Proactive notification to the client analyst when campaign metrics breach thresholds. '
        'Each alert should include: severity, client, campaign, ad set, '
        'recommended action, rationale, and confidence score.'
    ),
    'recommended_architecture': {
        'data_source': 'Meta Ads API (real-time) — not currently available to the team',
        'monitoring_logic': 'Compare CPL/CPT against benchmark using binary sigmoid P(Scale)',
        'severity_mapping': {
            'CRITICAL': 'P(Scale) < 0.20 — Pause zone — immediate action',
            'WARNING': 'P(Scale) 0.20-0.40 — Optimise zone — review and adjust',
            'INFO': 'P(Scale) > 0.70 — Scale Up zone — opportunity to increase budget',
            'NO_ALERT': 'P(Scale) 0.40-0.70 — Maintain zone — no action needed',
        },
        'delivery_channels': [
            'Email (SMTP / SendGrid)',
            'Slack (webhook)',
            'Dashboard (Streamlit / Retool)',
            'Google Sheets (simplest for the client analyst)',
        ],
    },
    'implementation_steps': [
        '1. Obtain Meta Ads API read access',
        '2. Schedule daily data pull',
        '3. Run binary XGBoost on fresh data',
        '4. Apply sigmoid-to-4-class mapping',
        '5. Generate alerts for Pause/Optimise campaigns',
        '6. Deliver via chosen channel',
    ],
    'constraints': [
        'Thresholds not yet validated with the client analyst/the client lead',
        'CLIENT_A model may be overconfident — flag all CLIENT_A alerts for manual review',
        'No Meta Ads API access — framework only, not live system',
    ],
}

# Export alerting documentation
with open(OUTPUT_DIR / 'alerting_system_recommendations.json', 'w') as f:
    json.dump(alerting_doc, f, indent=2)

print("\nAlerting system documentation exported to:")
print(f"  {OUTPUT_DIR / 'alerting_system_recommendations.json'}")


In [ ]:
# ── 14.2  Illustrative alert examples (from sigmoid data) ────────────────
#    These are NOT a live alerting system — they show what alerts WOULD
#    look like if the system were operational, using the sigmoid-derived
#    recommendations from Section 11.3.

print("\n--- Illustrative Alert Examples (what the system would produce) ---")

if sigmoid_client_a is not None:
    pause_campaigns = sigmoid_client_a[sigmoid_client_a['sigmoid_recommendation'] == 'Pause']
    optimise_campaigns = sigmoid_client_a[sigmoid_client_a['sigmoid_recommendation'] == 'Optimise']

    print(f"\nCLIENT_A — Campaigns that would trigger alerts:")
    print(f"  CRITICAL (Pause):   {len(pause_campaigns)} campaigns")
    print(f"  WARNING (Optimise): {len(optimise_campaigns)} campaigns")

    if len(pause_campaigns) > 0:
        print("\n  Sample CRITICAL alerts:")
        display_cols = [c for c in ['campaign_name', 'year_month', 'cpl_calc',
                                     'p_scale', 'sigmoid_recommendation'] if c in pause_campaigns.columns]
        print(pause_campaigns[display_cols].head(5).to_string(index=False))

if sigmoid_client_c is not None:
    pause_client_c = sigmoid_client_c[sigmoid_client_c['sigmoid_recommendation'] == 'Pause']
    optimise_client_c = sigmoid_client_c[sigmoid_client_c['sigmoid_recommendation'] == 'Optimise']

    print(f"\nCLIENT_C — Campaigns that would trigger alerts:")
    print(f"  CRITICAL (Pause):   {len(pause_client_c)} campaigns")
    print(f"  WARNING (Optimise): {len(optimise_client_c)} campaigns")

## 15. Baseline Metrics & Recommendations Report

### For the agency partner presentation

This section provides the baseline metrics and justification that the team
will present to the client lead and the client analyst. These are framed as consulting insights
based on the data analysis and model outputs.

## WINDBREAKER — BASELINE METRICS & RECOMMENDATIONS

EXECUTIVE SUMMARY
-----------------
Windbreaker analysed historical Meta Ads campaign data across 5 the agency partner
clients to build a machine learning-based recommendation engine.

TARGET METRICS (agreed with team, March 13):
  - CPL (Cost Per Lead) for lead-generation clients (Client A, Client D)
  - CPT (Cost Per Purchase) for e-commerce clients (Client C, Client B)
  - CPM for reach/awareness clients (Client E)

MODEL APPROACH:
  Binary XGBoost classifier with sigmoid probability output (advisor framework).
  P(Scale) mapped to 4 recommendations: Scale Up, Maintain, Optimise, Pause.
  Each recommendation includes a confidence score (P(Scale) value, 0-100%).
  4-class XGBoost retained as secondary comparison model.

In [ ]:
print("BASELINE BENCHMARKS")
print("-" * 50)
for key, value in benchmarks.items():
    if isinstance(value, (int, float)):
        print(f"  {key:<30} ${value:.2f}")
    else:
        print(f"  {key:<30} {value}")

### THRESHOLD FRAMEWORK

  **Rule-based thresholds (labelling):**
  Scale Up  : Cost metric < 70% of benchmark  (strong efficiency)
  Maintain  : Cost metric 70-110% of benchmark (on-target)
  Optimise  : Cost metric 110-160% of benchmark (above but recoverable)
  Pause     : Cost metric > 160% of benchmark  (poor return)

  **Sigmoid thresholds (binary model -> 4-class mapping, advisor framework):**
  Scale Up  : P(Scale) >= 0.70  (high probability of scaling)
  Maintain  : P(Scale) 0.40-0.70 (moderate, hold steady)
  Optimise  : P(Scale) 0.20-0.40 (below midpoint, needs work)
  Pause     : P(Scale) < 0.20  (low probability, poor return)

  Status: BOTH threshold sets are initial defaults.
  Neither has been validated with the client analyst or the client lead.
  Next step: Validate with client before any operational use.


In [ ]:
print(f"""
ALERTING SYSTEM
---------------
  Status: Documentation and recommendations only (per Week 9 decision).
  The team provides guidance on how to build the alerting system.
  See: alerting_system_recommendations.json

  Illustrative alerts from sigmoid data:
  CLIENT_A — Pause-zone campaigns:    {len(sigmoid_client_a[sigmoid_client_a['sigmoid_recommendation'] == 'Pause']) if sigmoid_client_a is not None else 'N/A'}
  CLIENT_A — Optimise-zone campaigns: {len(sigmoid_client_a[sigmoid_client_a['sigmoid_recommendation'] == 'Optimise']) if sigmoid_client_a is not None else 'N/A'}
  CLIENT_C — Pause-zone campaigns:   {len(sigmoid_client_c[sigmoid_client_c['sigmoid_recommendation'] == 'Pause']) if sigmoid_client_c is not None else 'N/A'}
  CLIENT_C — Optimise-zone campaigns: {len(sigmoid_client_c[sigmoid_client_c['sigmoid_recommendation'] == 'Optimise']) if sigmoid_client_c is not None else 'N/A'}
""")

The alerting system is provided as documentation and recommendations
  (per Week 9 decision). If implemented, it would generate proactive
  notifications when campaign metrics breach thresholds. Each alert would
  include severity level, affected client/campaign/ad set, recommended
  action with rationale, and model confidence score (P(Scale) value).

  Integration path: Can connect to email (SMTP/SendGrid),
  Slack (webhook), or a dashboard (Streamlit/Retool) for real-time
  delivery to the client analyst and the media buying team.

KEY FINDINGS

  1. Feature importance reveals which metrics actually drive campaign
     outcomes — this may differ from intuition and provides analytical
     value for media buying decisions.
  2. The binary sigmoid framework (the advisor's approach) provides a single
     P(Scale) value that maps to 4 recommendation tiers. This is more
     robust than direct 4-class prediction on small datasets.
  3. The Optimise sub-signals identify WHICH lever to pull (budget pacing,
     placement cost, or creative refresh) rather than just flagging a
     problem.
  4. Client A model may be overconfident due to small dataset — all Client A
     recommendations should be treated as indicative, not definitive.

## 16. Project Constraints & Documentation

### Purpose:
Document the constraints that shaped this project's scope and deliverables.
This protects the team by providing clear context for what was possible
within the given timeline and resources.



In [ ]:
print("\n" + "=" * 70)
print("  PROJECT CONSTRAINTS & DOCUMENTATION")
print("=" * 70)

constraints_doc = {
    'project': 'Windbreaker Recommendation Engine',
    'team': 'Bond University Capstone — DTSC71-304',
    'client': 'the agency partner (integrated growth agency)',
    'contacts': {
        'client_lead': 'the client lead',
        'media_buying': 'the client analyst',
        'academic_supervisor': 'Advisor',
    },
    'constraints': {
        'data_access': {
            'description': 'Daily-level data was requested from the client analyst on multiple occasions. '
                           'Daily CSV exports for CLIENT_A, CLIENT_C, and CLIENT_B were received on March 17 '
                           '(split by year/period). CLIENT_D and CLIENT_E daily xlsx files were '
                           'available from a previous batch. CLIENT_A daily data has sparse leads '
                           '(~1-2% of rows) so monthly data remains primary for CPL modelling.',
            'impact': 'CLIENT_C and CLIENT_B now have substantially more training data with purchase columns. '
                      'Day-of-week patterns and within-month trend detection are now possible for '
                      'all clients. CLIENT_A CPL model remains on monthly aggregates due to lead sparsity.',
            'mitigation': 'Dual-source pipeline: daily data as primary for CLIENT_C, CLIENT_B, CLIENT_D, CLIENT_E; '
                          'monthly as primary for CLIENT_A CPL model. CLIENT_A daily loaded for EDA/spend analysis.',
        },
        'api_access': {
            'description': 'Meta Ads API access was not provided or explained to the team. '
                           'All analysis relies on manual Excel exports from the client analyst.',
            'impact': 'Cannot build real-time data pipelines. Alerting system operates on '
                      'historical data rather than live campaign metrics.',
            'mitigation': 'Alerting system designed as a framework that can connect to live '
                          'data sources once API access is established.',
        },
        'threshold_validation': {
            'description': 'CPL/CPT thresholds (70/110/160%) are team-defined defaults based '
                           'on marketing industry standards. the client analyst and the client lead have not yet '
                           'formally validated these.',
            'impact': 'Model recommendations may not align with the agency partner\'s specific business '
                      'expectations for each client.',
            'mitigation': 'Thresholds are configurable. Validation session recommended as a '
                          'priority before any operational use.',
        },
        'data_volume': {
            'description': 'With daily data integration (v7): CLIENT_C and CLIENT_B now have thousands of rows '
                           '(significant upgrade from ~500 and ~21 monthly rows respectively). '
                           'CLIENT_A monthly ~100 rows for CPL. CLIENT_D and CLIENT_E daily volumes TBC.',
            'impact': 'CLIENT_C and CLIENT_B models are substantially more robust with daily data. '
                      'CLIENT_B moves from EDA-only to fully trainable. CLIENT_A CPL model remains '
                      'constrained by monthly sample size.',
            'mitigation': 'Used stratified K-Fold cross-validation. Binary model as PRIMARY '
                          '(per advisor framework — sigmoid P(Scale) mapped to 4 tiers). '
                          'Presented as POC / conversation catalyst, not production system.',
        },
        'timeline': {
            'description': 'Project scope changed multiple times due to evolving client '
                           'requirements and late data availability. Neural network approach '
                           'was abandoned in favour of XGBoost based on mentor guidance '
                           '(March 13).',
            'impact': 'Limited time for model refinement and iterative testing.',
            'mitigation': 'Focused on delivering a working POC with clear documentation '
                          'and an integration path for future development.',
        },
    },
    'recommendations_for_next_phase': [
        'Validate CPL/CPT thresholds with the client analyst and the client lead for each client.',
        'Validate sigmoid probability thresholds (0.70/0.40/0.20) with client.',
        'Obtain Meta Ads API access for real-time data pipeline.',
        'Build alerting system per documentation in alerting_system_recommendations.json.',
        'Build a simple dashboard (Streamlit or Google Sheets) for the client analyst.',
        'Conduct A/B testing: compare model recommendations vs the client analyst\'s decisions.',
        'Follow up with the client analyst for CLIENT_A daily data with more granular lead attribution.',
        'Expand CLIENT_D and CLIENT_E training data if daily volumes are sufficient.',
        'Consider KNN imputation for missing values (discussed with the advisor, not yet implemented).',
    ],
    'generated': datetime.now().strftime('%Y-%m-%d %H:%M'),
}

# Export constraints document
with open(OUTPUT_DIR / 'project_constraints.json', 'w') as f:
    json.dump(constraints_doc, f, indent=2)

# Print recommendations
print("\nRECOMMENDATIONS FOR NEXT PHASE:")
for i, rec in enumerate(constraints_doc['recommendations_for_next_phase'], 1):
    print(f"  {i}. {rec}")

print(f"\nConstraints documentation exported to: {OUTPUT_DIR / 'project_constraints.json'}")

# ── Final export summary ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  ALL EXPORTED FILES")
print("=" * 70)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:<40} ({f.stat().st_size / 1024:.1f} KB)")

## v10 Summary

### What was delivered in v8:

**Phase 1 (Data — Sections 1-10):**
- Dual-source data loading: daily CSVs (Client C, Client B, Client A) + daily xlsx (Client D, Client E) + monthly xlsx (Client A)
- Client A monthly as primary for CPL model; Client A daily for supplementary EDA
- Client C and Client B daily data with purchase columns (previously unavailable)
- Automatic derivation of Cost per Purchase for Client C (not in export)
- Granularity-aware date parsing (Day column for daily, Month column for monthly)
- Day-of-week features for daily data (supports the agency partner's optimisation signal)
- Parse KPI guide and identify critical metrics
- EDA deep dive for all 5 clients
- Benchmark CPL (leads) and CPT (purchases)
- Assign 4-class and binary recommendation labels
- Engineer Optimise sub-signals
- Build feature matrices (KPI-guide-filtered, granularity-aware rolling)

**Phase 2 (Modelling — Sections 11-12):**
- **Binary XGBoost as PRIMARY** — sigmoid P(Scale) mapped to 4 tiers (the advisor's framework)
- **4-class XGBoost as SECONDARY** — comparison only, NOT primary deliverable
- Sigmoid-to-4-class mapping function with configurable thresholds
- Comparison: sigmoid-derived labels vs rule-based labels
- Stratified K-Fold cross-validation
- **F1 Macro** (per advisor agreement, treats all classes equally)
- Feature importance from BOTH binary and 4-class models
- Cross-model comparison (Client A vs Client C)
- 7-day rolling averages for daily data, 3-month for monthly

**Phase 3 (Alerting & Recommendations — Sections 13-16):**
- **Confidence validation** using binary sigmoid P(Scale) (NOT 4-class softmax)
- Client A overconfidence documented and flagged
- **Alerting system as DOCUMENTATION ONLY** (per Week 9 key decision)
- Alerting recommendations exported as JSON guidance for the agency partner
- Illustrative alert examples from sigmoid data (not a live system)
- Baseline metrics and recommendations report for the agency partner presentation
- **Imputation method documented** (median fill, KNN discussed but not used)
- Project constraints documentation updated

### What changed from v7:
1. **Binary-first approach** — the advisor's sigmoid-to-4-class mapping is now PRIMARY
2. **4-class demoted to comparison** — no longer the main deliverable
3. **F1 Macro** — changed from F1 Weighted to match advisor agreement
4. **Feature importance from binary model** — primary extraction source
5. **Section 13 rewritten** — validates binary sigmoid confidence, not 4-class softmax
6. **Section 14 rewritten** — documentation/recommendations only per Week 9 decision
7. **Imputation method clarified** — median fill documented (KNN noted as discussed)
8. **Client A overconfidence handling** — explicitly flagged in confidence validation

### Data Status
| Client | Source | Granularity | Training Status |
|--------|--------|-------------|-----------------|
| Client A | Monthly xlsx | Monthly | Full training (CPL) |
| Client C | Daily CSVs (4 files) | Daily | Full training (CPT) |
| Client B | Daily CSVs (4 files) | Daily | Full training (CPT) — upgraded from EDA-only |
| Client D | Daily xlsx | Daily | Evaluate volume |
| Client E | Daily xlsx | Daily | EDA + validation |
| Client A (EDA) | Daily CSVs (8 files) | Daily | Spend/click analysis only (leads sparse) |

| Note imputed data is simulation | Advisor, Week 9 | Done (v8) |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  RESULTS EXPORT
# ═══════════════════════════════════════════════════════════════════════════

import json
from datetime import datetime

print("=" * 80)
print("  WINDBREAKER v8 — FULL RESULTS EXPORT")
print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 80)

# ── 1. Data volumes ──────────────────────────────────────────────────────
print("\n\n### 1. DATA VOLUMES ###")
for name, df in cleaned.items():
    gran = df['_granularity'].iloc[0] if len(df) > 0 else 'N/A'
    nulls_pct = df.isnull().mean().mean() * 100
    print(f"  {name:<10} {gran:<10} {len(df):>6} rows  |  avg null: {nulls_pct:.1f}%")

# ── 2. Benchmarks ────────────────────────────────────────────────────────
print("\n\n### 2. BENCHMARKS ###")
for key, value in benchmarks.items():
    if isinstance(value, (int, float)):
        print(f"  {key:<30} ${value:.2f}")
    else:
        print(f"  {key:<30} {value}")

# ── 3. Rule-based label distributions ────────────────────────────────────
print("\n\n### 3. RULE-BASED LABEL DISTRIBUTIONS ###")
print("\nCLIENT_A (4-class):")
print(client_a_labelled['recommendation'].value_counts().to_string())
print(f"\nCLIENT_A (binary):")
print(client_a_labelled['recommendation_binary'].value_counts().to_string())
if 'recommendation' in client_c_labelled.columns and client_c_labelled['recommendation'].notna().any():
    print(f"\nCLIENT_C (4-class):")
    print(client_c_labelled['recommendation'].value_counts().to_string())
    print(f"\nCLIENT_C (binary):")
    print(client_c_labelled['recommendation_binary'].value_counts().to_string())

# ── 4. Binary model results (PRIMARY) ────────────────────────────────────
print("\n\n### 4. BINARY MODEL RESULTS [PRIMARY] ###")
if model_client_a_bin is not None:
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
    print("\nCLIENT_A Binary — Confusion Matrix:")
    cm = confusion_matrix(y_client_a_bin_clean, pred_client_a_bin, labels=classes_client_a_bin)
    print(f"  Labels: {list(classes_client_a_bin)}")
    print(f"  {cm.tolist()}")
    print(f"\nCLIENT_A Binary — Full Classification Report:")
    print(classification_report(y_client_a_bin_clean, pred_client_a_bin, zero_division=0))

if model_client_c_bin is not None:
    print("\nCLIENT_C Binary — Confusion Matrix:")
    cm_e = confusion_matrix(y_client_c_bin_clean, pred_client_c_bin, labels=classes_client_c_bin)
    print(f"  Labels: {list(classes_client_c_bin)}")
    print(f"  {cm_e.tolist()}")
    print(f"\nCLIENT_C Binary — Full Classification Report:")
    print(classification_report(y_client_c_bin_clean, pred_client_c_bin, zero_division=0))

# ── 5. Sigmoid mapping results ───────────────────────────────────────────
print("\n\n### 5. SIGMOID-TO-4-CLASS MAPPING ###")
print(f"\nThresholds: {SIGMOID_THRESHOLDS}")

if sigmoid_client_a is not None:
    print(f"\nCLIENT_A — Sigmoid distribution:")
    print(sigmoid_client_a['sigmoid_recommendation'].value_counts().to_string())
    print(f"\nCLIENT_A — P(Scale) stats:")
    print(sigmoid_client_a['p_scale'].describe().round(4).to_string())
    print(f"\nCLIENT_A — P(Scale) by recommendation:")
    print(sigmoid_client_a.groupby('sigmoid_recommendation')['p_scale'].agg(
        ['count', 'mean', 'std', 'min', 'max']).round(4).to_string())

if sigmoid_client_c is not None:
    print(f"\nCLIENT_C — Sigmoid distribution:")
    print(sigmoid_client_c['sigmoid_recommendation'].value_counts().to_string())
    print(f"\nCLIENT_C — P(Scale) stats:")
    print(sigmoid_client_c['p_scale'].describe().round(4).to_string())
    print(f"\nCLIENT_C — P(Scale) by recommendation:")
    print(sigmoid_client_c.groupby('sigmoid_recommendation')['p_scale'].agg(
        ['count', 'mean', 'std', 'min', 'max']).round(4).to_string())

# ── 6. Sigmoid vs Rule-based agreement ───────────────────────────────────
print("\n\n### 6. SIGMOID vs RULE-BASED AGREEMENT ###")
if sigmoid_client_a is not None:
    sigmoid_client_a['_match'] = sigmoid_client_a['sigmoid_recommendation'] == sigmoid_client_a['rule_based_label']
    print(f"\nCLIENT_A — Overall agreement: {sigmoid_client_a['_match'].mean():.1%}")
    print("\nCLIENT_A — Cross-tabulation (rows=rule-based, cols=sigmoid):")
    ct = pd.crosstab(sigmoid_client_a['rule_based_label'], sigmoid_client_a['sigmoid_recommendation'])
    print(ct.to_string())

if sigmoid_client_c is not None:
    sigmoid_client_c['_match'] = sigmoid_client_c['sigmoid_recommendation'] == sigmoid_client_c['rule_based_label']
    print(f"\nCLIENT_C — Overall agreement: {sigmoid_client_c['_match'].mean():.1%}")
    print("\nCLIENT_C — Cross-tabulation (rows=rule-based, cols=sigmoid):")
    ct_e = pd.crosstab(sigmoid_client_c['rule_based_label'], sigmoid_client_c['sigmoid_recommendation'])
    print(ct_e.to_string())

# ── 7. 4-class model results (COMPARISON) ────────────────────────────────
print("\n\n### 7. 4-CLASS MODEL RESULTS [COMPARISON] ###")
print("\nCLIENT_A 4-Class — Confusion Matrix:")
cm4 = confusion_matrix(y_client_a_clean, pred_client_a_4c, labels=classes_client_a_4c)
print(f"  Labels: {list(classes_client_a_4c)}")
print(f"  {cm4.tolist()}")
print(f"\nCLIENT_A 4-Class — Classification Report:")
print(classification_report(y_client_a_clean, pred_client_a_4c, zero_division=0))

if model_client_c_4c is not None:
    print("\nCLIENT_C 4-Class — Confusion Matrix:")
    cm4_e = confusion_matrix(y_client_c_clean, pred_client_c_4c, labels=classes_client_c_4c)
    print(f"  Labels: {list(classes_client_c_4c)}")
    print(f"  {cm4_e.tolist()}")
    print(f"\nCLIENT_C 4-Class — Classification Report:")
    print(classification_report(y_client_c_clean, pred_client_c_4c, zero_division=0))

# ── 8. Feature importance (all models) ───────────────────────────────────
print("\n\n### 8. FEATURE IMPORTANCE ###")
if model_client_a_bin is not None:
    print("\nCLIENT_A Binary [PRIMARY]:")
    fi_client_a = pd.DataFrame({'feature': X_client_a_clean.columns,
                           'importance': model_client_a_bin.feature_importances_})
    fi_client_a = fi_client_a.sort_values('importance', ascending=False)
    print(fi_client_a.to_string(index=False))

if model_client_c_bin is not None:
    print("\nCLIENT_C Binary [PRIMARY]:")
    fi_client_c = pd.DataFrame({'feature': X_client_c_clean.columns,
                            'importance': model_client_c_bin.feature_importances_})
    fi_client_c = fi_client_c.sort_values('importance', ascending=False)
    print(fi_client_c.to_string(index=False))

print("\nCLIENT_A 4-Class [COMPARISON]:")
fi_client_a4 = pd.DataFrame({'feature': X_client_a_clean_4c.columns,
                         'importance': model_client_a_4c.feature_importances_})
fi_client_a4 = fi_client_a4.sort_values('importance', ascending=False)
print(fi_client_a4.to_string(index=False))

if model_client_c_4c is not None:
    print("\nCLIENT_C 4-Class [COMPARISON]:")
    fi_client_c4 = pd.DataFrame({'feature': X_client_c_clean_4c.columns,
                             'importance': model_client_c_4c.feature_importances_})
    fi_client_c4 = fi_client_c4.sort_values('importance', ascending=False)
    print(fi_client_c4.to_string(index=False))

# ── 9. Confidence validation ─────────────────────────────────────────────
print("\n\n### 9. CONFIDENCE VALIDATION ###")
if sigmoid_client_a is not None and 'matches_rule' in sigmoid_client_a.columns:
    print("\nCLIENT_A — Agreement by confidence level:")
    ca = sigmoid_client_a.groupby('confidence_level')['matches_rule'].agg(['mean', 'count'])
    ca.columns = ['agreement_rate', 'n_rows']
    print(ca.to_string())

if sigmoid_client_c is not None and 'matches_rule' in sigmoid_client_c.columns:
    print("\nCLIENT_C — Agreement by confidence level:")
    ca_e = sigmoid_client_c.groupby('confidence_level')['matches_rule'].agg(['mean', 'count'])
    ca_e.columns = ['agreement_rate', 'n_rows']
    print(ca_e.to_string())

# ── 10. Sample predictions ───────────────────────────────────────────────
print("\n\n### 10. SAMPLE PREDICTIONS (first 15 rows) ###")
if sigmoid_client_a is not None:
    sample_cols = [c for c in ['campaign_name', 'year_month', 'cpl_calc',
                               'p_scale', 'sigmoid_recommendation',
                               'rule_based_label', 'confidence_level']
                   if c in sigmoid_client_a.columns]
    print("\nCLIENT_A — Sample:")
    print(sigmoid_client_a[sample_cols].head(15).to_string(index=False))

if sigmoid_client_c is not None:
    sample_cols_e = [c for c in ['campaign_name', 'year_month', 'cost_per_purchase',
                                  'p_scale', 'sigmoid_recommendation',
                                  'rule_based_label', 'confidence_level']
                     if c in sigmoid_client_c.columns]
    print("\nCLIENT_C — Sample:")
    print(sigmoid_client_c[sample_cols_e].head(15).to_string(index=False))

print("\n\n" + "=" * 80)
print("  END OF EXPORT")
print("=" * 80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  WINDBREAKER v10 — DATA PIPELINE DIAGNOSTIC
#  Purpose: Trace exactly where CLIENT_A rows are lost from raw → model
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

print("=" * 80)
print("  WINDBREAKER v10 — DATA PIPELINE DIAGNOSTIC")
print("  Tracing every row from raw data to model input")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════
# PART A — RAW DATA INVENTORY (what was loaded before any cleaning)
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART A: RAW DATA INVENTORY ###")
print("(What was loaded before any cleaning)\n")

for name, df in raw.items():
    gran = df['_granularity'].iloc[0] if '_granularity' in df.columns and len(df) > 0 else 'unknown'
    print(f"  {name:<12} {gran:<10} {len(df):>6} raw rows x {len(df.columns)} cols")

# CLIENT_A daily EDA data (loaded separately)
if 'client_a_daily_eda' in dir() and len(client_a_daily_eda) > 0:
    print(f"  {'CLIENT_A_daily':<12} {'daily':<10} {len(client_a_daily_eda):>6} raw rows  [EDA/spend only]")
elif 'client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0:
    print(f"  {'CLIENT_A_daily':<12} {'daily':<10} {len(client_a_daily_cleaned):>6} rows  [EDA/spend only, post-clean]")
else:
    print(f"  {'CLIENT_A_daily':<12} NOT LOADED or empty")

# ═══════════════════════════════════════════════════════════════════════════
# PART B — POST-CLEANING INVENTORY
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART B: POST-CLEANING INVENTORY ###")
print("(After clean_client() — nulls removed, zero-spend dropped)\n")

for name, df in cleaned.items():
    gran = df['_granularity'].iloc[0] if '_granularity' in df.columns and len(df) > 0 else 'unknown'
    raw_count = len(raw[name]) if name in raw else 0
    lost = raw_count - len(df)
    pct_lost = (lost / raw_count * 100) if raw_count > 0 else 0
    print(f"  {name:<12} {gran:<10} {len(df):>6} cleaned rows  (lost {lost} = {pct_lost:.1f}% from raw)")

# ═══════════════════════════════════════════════════════════════════════════
# PART C — CLIENT_A DEEP DIVE: WHERE ARE THE ROWS GOING?
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART C: CLIENT_A DEEP DIVE — ROW-BY-ROW LOSS TRACE ###\n")

client_a_raw = raw.get('CLIENT_A', pd.DataFrame())
client_a_clean = cleaned.get('CLIENT_A', pd.DataFrame())

print(f"Step 0 — Raw loaded:                    {len(client_a_raw)} rows")
print(f"         Granularity:                    {client_a_raw['_granularity'].iloc[0] if len(client_a_raw) > 0 else 'N/A'}")

# Check what date column exists
has_month = 'Month' in client_a_raw.columns or 'month' in client_a_raw.columns
has_day = 'Day' in client_a_raw.columns or 'day' in client_a_raw.columns
print(f"         Has 'Month' column:             {has_month}")
print(f"         Has 'Day' column:               {has_day}")
print(f"         Column count:                   {len(client_a_raw.columns)}")

# After cleaning
print(f"\nStep 1 — After clean_client():           {len(client_a_clean)} rows")

# Check spend column
if 'spend' in client_a_clean.columns:
    zero_spend = (client_a_clean['spend'] == 0).sum()
    null_spend = client_a_clean['spend'].isna().sum()
    pos_spend = (client_a_clean['spend'] > 0).sum()
    print(f"         Spend > 0:                      {pos_spend}")
    print(f"         Spend = 0:                       {zero_spend}")
    print(f"         Spend = NaN:                     {null_spend}")

# Check leads column
for leads_col in ['all_leads', 'leads', 'Leads', 'All Leads']:
    if leads_col in client_a_clean.columns:
        leads = pd.to_numeric(client_a_clean[leads_col], errors='coerce')
        total_leads = leads.sum()
        rows_with_leads = (leads > 0).sum()
        rows_no_leads = (leads == 0).sum() + leads.isna().sum()
        print(f"\nStep 2 — Lead filtering (column: '{leads_col}'):")
        print(f"         Total leads in data:            {total_leads:.0f}")
        print(f"         Rows with leads > 0:            {rows_with_leads}")
        print(f"         Rows with leads = 0 or NaN:     {rows_no_leads}")
        print(f"         >>> THIS IS THE MAIN FILTER <<<")
        break

# Check what client_a_labelled looks like
if 'client_a_labelled' in dir():
    print(f"\nStep 3 — client_a_labelled (after CPL calc):  {len(client_a_labelled)} rows")
    if 'recommendation' in client_a_labelled.columns:
        print(f"         Has recommendation labels:      {client_a_labelled['recommendation'].notna().sum()}")
        print(f"         NaN labels:                     {client_a_labelled['recommendation'].isna().sum()}")
    if 'cpl_calc' in client_a_labelled.columns:
        print(f"         CPL stats:")
        print(f"           Mean:   ${client_a_labelled['cpl_calc'].mean():.2f}")
        print(f"           Median: ${client_a_labelled['cpl_calc'].median():.2f}")
        print(f"           Min:    ${client_a_labelled['cpl_calc'].min():.2f}")
        print(f"           Max:    ${client_a_labelled['cpl_calc'].max():.2f}")

# Check feature engineering output
if 'X_client_a' in dir():
    print(f"\nStep 4 — X_client_a (feature matrix):         {len(X_client_a)} rows x {X_client_a.shape[1]} features")

# Check what goes into the model
if 'X_client_a_clean' in dir() and X_client_a_clean is not None:
    print(f"\nStep 5 — X_client_a_clean (model input):      {len(X_client_a_clean)} rows")
    print(f"         y labels non-null:               {y_client_a_bin_clean.notna().sum() if 'y_client_a_bin_clean' in dir() and y_client_a_bin_clean is not None else 'N/A'}")

# ═══════════════════════════════════════════════════════════════════════════
# PART D — CLIENT_A DAILY vs MONTHLY COMPARISON
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART D: CLIENT_A DAILY vs MONTHLY COMPARISON ###\n")

print("CLIENT_A daily CSVs are now PRIMARY for the CPL model (986 rows with leads).")
print("CLIENT_A monthly xlsx retained as comparison only.")
print()

# Monthly details
if len(client_a_clean) > 0:
    print(f"CLIENT_A MONTHLY (comparison only — daily is now primary):")
    print(f"  Rows:          {len(client_a_clean)}")
    if 'date_start' in client_a_clean.columns:
        print(f"  Date range:    {client_a_clean['date_start'].min()} to {client_a_clean['date_start'].max()}")
    if 'campaign_name' in client_a_clean.columns:
        print(f"  Unique campaigns: {client_a_clean['campaign_name'].nunique()}")
    if 'all_leads' in client_a_clean.columns:
        leads = pd.to_numeric(client_a_clean['all_leads'], errors='coerce')
        print(f"  Total leads:      {leads.sum():.0f}")
        print(f"  Rows with leads:  {(leads > 0).sum()} out of {len(client_a_clean)} ({(leads > 0).mean():.1%})")
        print(f"  Rows WITHOUT leads: {(leads <= 0).sum() + leads.isna().sum()}")

# Daily details
if 'client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0:
    client_a_d = client_a_daily_cleaned
    print(f"\nCLIENT_A DAILY (PRIMARY for CPL model — 986 rows with leads):")
    print(f"  Rows:          {len(client_a_d)}")
    if 'date_start' in client_a_d.columns:
        print(f"  Date range:    {client_a_d['date_start'].min()} to {client_a_d['date_start'].max()}")
    if 'campaign_name' in client_a_d.columns:
        print(f"  Unique campaigns: {client_a_d['campaign_name'].nunique()}")
    for leads_col in ['all_leads', 'leads']:
        if leads_col in client_a_d.columns:
            leads_d = pd.to_numeric(client_a_d[leads_col], errors='coerce')
            print(f"  Total leads:      {leads_d.sum():.0f}")
            print(f"  Rows with leads:  {(leads_d > 0).sum()} out of {len(client_a_d)} ({(leads_d > 0).mean():.1%})")
            print(f"  Rows WITHOUT leads: {(leads_d <= 0).sum() + leads_d.isna().sum()}")
            break
    if 'spend' in client_a_d.columns:
        print(f"  Total spend:      ${client_a_d['spend'].sum():,.2f}")

# ═══════════════════════════════════════════════════════════════════════════
# PART E — ALL CLIENTS: SAME DIAGNOSTIC
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART E: ALL CLIENTS — MODELLING PIPELINE LOSS ###\n")

print(f"{'Client':<12} {'Raw':>8} {'Cleaned':>8} {'Labelled':>8} {'Model':>8} {'Loss %':>8}")
print("-" * 60)

# CLIENT_A
client_a_lab_n = len(client_a_labelled) if 'client_a_labelled' in dir() else 0
client_a_model_n = len(X_client_a_clean) if 'X_client_a_clean' in dir() and X_client_a_clean is not None else 0
client_a_daily_n = len(client_a_daily_cleaned) if 'client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0 else 0
client_a_loss = (1 - client_a_model_n / client_a_daily_n) * 100 if client_a_daily_n > 0 else 0
print(f"{'CLIENT_A':<12} {client_a_daily_n:>8} {client_a_daily_n:>8} {client_a_lab_n:>8} {client_a_model_n:>8} {client_a_loss:>7.1f}%")
print(f"{'CLIENT_A_monthly':<12} {len(raw['CLIENT_A']):>8} {len(cleaned['CLIENT_A']):>8} {'84':>8} {'comparison':>8} {'N/A':>8}")

# CLIENT_C
client_c_lab_n = len(client_c_labelled) if 'client_c_labelled' in dir() else 0
client_c_model_n = len(X_client_c_clean) if 'X_client_c_clean' in dir() and X_client_c_clean is not None else 0
client_c_loss = (1 - client_c_model_n / len(raw['CLIENT_C'])) * 100 if len(raw['CLIENT_C']) > 0 else 0
print(f"{'CLIENT_C':<12} {len(raw['CLIENT_C']):>8} {len(cleaned['CLIENT_C']):>8} {client_c_lab_n:>8} {client_c_model_n:>8} {client_c_loss:>7.1f}%")

# CLIENT_B, CLIENT_D, CLIENT_E — note these may not have labelled/model data
for name in ['CLIENT_B', 'CLIENT_D', 'CLIENT_E']:
    raw_n = len(raw[name]) if name in raw else 0
    clean_n = len(cleaned[name]) if name in cleaned else 0
    print(f"{name:<12} {raw_n:>8} {clean_n:>8} {'N/A':>8} {'N/A':>8} {'N/A':>8}")

# ═══════════════════════════════════════════════════════════════════════════
# PART F — NULL ANALYSIS PER CLIENT (which columns have the most damage)
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART F: NULL ANALYSIS — KEY COLUMNS ###\n")

key_cols = ['spend', 'impressions', 'reach', 'frequency', 'cpm',
            'ctr_link', 'cpc_link', 'link_clicks', 'all_leads',
            'cost_per_lead', 'purchases', 'cost_per_purchase', 'roas']

for name, df in cleaned.items():
    print(f"\n{name} ({len(df)} rows):")
    for col in key_cols:
        if col in df.columns:
            n_null = df[col].isna().sum()
            n_zero = (df[col] == 0).sum() if df[col].dtype in ['float64', 'int64'] else 0
            n_usable = len(df) - n_null - n_zero
            pct_usable = n_usable / len(df) * 100 if len(df) > 0 else 0
            flag = " <<<" if pct_usable < 50 else ""
            print(f"  {col:<25} usable: {n_usable:>6} ({pct_usable:>5.1f}%)  null: {n_null:>5}  zero: {n_zero:>5}{flag}")

# ═══════════════════════════════════════════════════════════════════════════
# PART G — COULD CLIENT_A DAILY REPLACE MONTHLY?
# ═══════════════════════════════════════════════════════════════════════════

print("\n\n### PART G: COULD CLIENT_A DAILY DATA BE USED FOR MODELLING? ###\n")

if 'client_a_daily_cleaned' in dir() and len(client_a_daily_cleaned) > 0:
    client_a_d = client_a_daily_cleaned
    print(f"CLIENT_A daily has {len(client_a_d)} rows.")

    # Check lead sparsity
    for leads_col in ['all_leads', 'leads']:
        if leads_col in client_a_d.columns:
            leads_d = pd.to_numeric(client_a_d[leads_col], errors='coerce')
            rows_with = (leads_d > 0).sum()
            pct_with = rows_with / len(client_a_d) * 100
            print(f"\nLead sparsity check (column: '{leads_col}'):")
            print(f"  Rows with leads > 0:   {rows_with} out of {len(client_a_d)} ({pct_with:.1f}%)")

            if rows_with > 0:
                client_a_d_leads = client_a_d[leads_d > 0].copy()
                client_a_d_leads['cpl_daily'] = client_a_d_leads['spend'] / leads_d[leads_d > 0]
                print(f"  If we filtered to leads > 0: {rows_with} rows for model")
                print(f"  Daily CPL stats:")
                print(f"    Mean:   ${client_a_d_leads['cpl_daily'].mean():.2f}")
                print(f"    Median: ${client_a_d_leads['cpl_daily'].median():.2f}")
                print(f"    Min:    ${client_a_d_leads['cpl_daily'].min():.2f}")
                print(f"    Max:    ${client_a_d_leads['cpl_daily'].max():.2f}")

                if rows_with >= 50:
                    print(f"\n  >>> {rows_with} rows IS enough for a binary model.")
                elif rows_with >= 20:
                    print(f"\n  >>> {rows_with} rows is marginal. Binary model might work")
                    print(f"  >>> but K-Fold CV would need reduced folds (2-3).")
                else:
                    print(f"\n  >>> {rows_with} rows is too few. Monthly remains the only option.")
            break
    else:
        print("  No leads column found in CLIENT_A daily data.")
else:
    print("CLIENT_A daily data not available in this notebook session.")
    print("(It may not have been loaded, or the variable name differs.)")

print("\n\n" + "=" * 80)
print("  END OF DIAGNOSTIC")
print("=" * 80)